# 🧠 NeuroVerse — Motor Spiral & Wave Drawing Model Training
## Google Colab Notebook: Parkinson's Disease Detection from Spiral/Wave Drawings

### Clinical Background
**Spiral and wave drawing tests** are established clinical tools for detecting Parkinson's Disease (PD).
PD patients exhibit characteristic drawing impairments:
- **Tremor** — 4-6 Hz oscillations visible as wavy/jagged lines
- **Micrographia** — progressively smaller drawing amplitude
- **Bradykinesia** — slower drawing speed, uneven pressure
- **Rigidity** — loss of smooth curvature, angular deviations

### Datasets (Three Sources — Spiral & Wave Only)

| # | Source | Type | ~Images | Classes |
|---|--------|------|---------|---------|
| 1 | **YOLODatasetFull_Augmented** | Spiral + Wave | ~11,786 | healthy spiral/wave, parkinson spiral/wave |
| 2 | **HandPD_Augmented_Data** (Pereira et al.) | Spiral (3 augmentation types) | ~8,571 | HealthySpiral, PatientSpiral |
| 3 | **UCI Digitized Graphics Tablet** (Isenkul et al.) | Tablet CSV → Rendered Spirals | ~hundreds | control, parkinson |

**Combined: ~20,000+ real spiral/wave images**

### Architecture
- **EfficientNet-B0** (pretrained ImageNet) + custom PD classification head
- **5 XAI Methods**: GradCAM, GradCAM++, LIME, Integrated Gradients, Occlusion Sensitivity

### XAI (Explainable AI) Methods
| Method | Type | What It Shows |
|--------|------|---------------|
| **GradCAM** | Gradient-based | Coarse heatmap from last conv layer |
| **GradCAM++** | Gradient-based | Better multi-object localization |
| **LIME** | Perturbation-based | Superpixel importance (model-agnostic) |
| **Integrated Gradients** | Attribution | Pixel-level feature importance |
| **Occlusion Sensitivity** | Perturbation-based | Sliding-window region importance |

### References
1. Pereira, C.R. et al. (2016). *"A new computer vision approach for the early diagnosis of PD."* ESWA.
2. Zham, P. et al. (2017). *"Distinguishing PD Stages Using Composite Index."* Frontiers in Neurology.
3. Selvaraju, R.R. et al. (2017). *"Grad-CAM: Visual Explanations from Deep Networks."* ICCV.
4. Ribeiro, M.T. et al. (2016). *"Why Should I Trust You? Explaining the Predictions of Any Classifier."* KDD.
5. Sundararajan, M. et al. (2017). *"Axiomatic Attribution for Deep Networks."* ICML.
6. Chattopadhay, A. et al. (2018). *"Grad-CAM++: Generalized Gradient-based Visual Explanations."* WACV.

### Output
- `motor_spiral_model.pt` — PyTorch model for NeuroVerse backend
- UPDRS-aligned tremor severity scoring
- Multi-XAI visualizations for doctor dashboard

## 1️⃣ Environment Setup & Dataset Upload (Google Colab)

### ⚠️ IMPORTANT — Upload Datasets BEFORE Running!

**Upload these zip files to Colab** (drag & drop into the 📁 Files panel on the left sidebar):
1. `YOLODatasetFull_Augmented.zip` — ~11,786 spiral + wave images
2. `HandPD_Spiral.zip` — ~8,571 spiral images
3. `UCI_PD_Spiral.zip` — Digitized tablet CSV data (optional)

### How to prepare your zip files:
```
YOLODatasetFull_Augmented.zip
  └── YOLODatasetFull/
       ├── images/train/    (healthy_*.png, parkinson_*.png ~9,431)
       ├── images/val/      (~2,355)
       ├── labels/train/    (YOLO format .txt)
       ├── labels/val/
       └── dataset.yaml

HandPD_Spiral.zip
  └── HandPD_Augmented_Data/
       ├── Geometric_Augmentation/{training_set,test_set}/spiral/{HealthySpiral,PatientSpiral}/
       ├── Mixed_Augmentation/{training_set,test_set}/spiral/{HealthySpiral,PatientSpiral}/
       └── Photometric_Augmentation/{training_set,test_set}/spiral/{aug_healthyspiral,aug_patientspiral}/

UCI_PD_Spiral.zip (optional)
  └── hw_dataset/{control,parkinson}/*.txt
```

### This notebook trains ONLY on spiral + wave drawings.
For meander drawings, use the separate **motor_meander_training.ipynb** notebook.

In [ ]:
# ============================================================
# CELL 1: Install Dependencies & Load Data
# ============================================================
# 📌 DATASETS on Google Drive → My Drive/Neuro_Datasets/
#    1. YOLODatasetFull_Augmented.zip                                  (~11,786 images)
#    2. HandPD_Spiral.zip                                               (~8,571 images)
#    3. parkinson+disease+spiral+drawings+using+digitized+graphics+tablet (1).zip  (Kaggle tablet)

# ── Install packages (Colab-safe — does NOT downgrade numpy) ──
# ⚠️ If you previously ran a cell that downgraded numpy, RESTART RUNTIME first!
#    (Runtime → Restart runtime, then re-run this cell)
!pip install -q timm grad-cam lime
!pip install -q captum --no-deps   # captum works with numpy 2.x, its pin is overly strict

import os, glob, json, hashlib, warnings, random, zipfile, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from PIL import Image, ImageDraw

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    f1_score, accuracy_score
)

import timm

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════
# Reproducibility
# ═══════════════════════════════════════════════════════════
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# ═══════════════════════════════════════════════════════════
# Output directories
# ═══════════════════════════════════════════════════════════
OUTPUT_DIR = '/content/motor_spiral_output'
MODEL_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
BASE_DIR = '/content/datasets'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'export'), exist_ok=True)

# ═══════════════════════════════════════════════════════════
# 📂 Mount Google Drive & copy zips to local disk
# ═══════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FOLDER = "/content/drive/MyDrive/Neuro_Datasets"   # ← your Drive folder
DATA_DIR = '/content/datasets'
os.makedirs(DATA_DIR, exist_ok=True)

# Exact zip filenames as they appear in your Drive folder
ZIPS = {
    'YOLODatasetFull_Augmented': os.path.join(DRIVE_FOLDER, 'YOLODatasetFull_Augmented.zip'),
    'HandPD_Spiral':             os.path.join(DRIVE_FOLDER, 'HandPD_Spiral.zip'),
    'UCI_PD_Spiral':             os.path.join(DRIVE_FOLDER, 'parkinson+disease+spiral+drawings+using+digitized+graphics+tablet (1).zip'),
}

def extract_if_needed(zip_path, dest_dir, name):
    """Copy from Drive to local disk then extract (faster than extracting from Drive directly)."""
    marker = os.path.join(dest_dir, f'.{name}_extracted')
    if os.path.exists(marker):
        print(f"   ✅ {name} already extracted")
        return True
    if not os.path.exists(zip_path):
        print(f"   ⚠️  {name} not found at {zip_path} — skipping")
        return False
    # Copy to local disk first for faster extraction
    local_zip = os.path.join('/content', os.path.basename(zip_path))
    if not os.path.exists(local_zip):
        print(f"   📋 Copying {name} from Drive to local disk...")
        shutil.copy2(zip_path, local_zip)
    size_mb = os.path.getsize(local_zip) / 1e6
    print(f"   📦 Extracting {name} ({size_mb:.0f} MB)...")
    with zipfile.ZipFile(local_zip, 'r') as z:
        z.extractall(dest_dir)
    open(marker, 'w').close()
    os.remove(local_zip)  # Free local disk space
    print(f"   ✅ {name} extracted!")
    return True

# Verify required zips exist on Drive
print("\n📂 Checking Drive folder:", DRIVE_FOLDER)
if os.path.exists(DRIVE_FOLDER):
    drive_files = os.listdir(DRIVE_FOLDER)
    print(f"   Files found: {len(drive_files)}")
    for f in drive_files:
        print(f"   • {f}")
else:
    print(f"   ❌ Drive folder not found: {DRIVE_FOLDER}")
    print(f"   Make sure your folder is named 'Neuro_Datasets' in My Drive")

print()
assert os.path.exists(ZIPS['YOLODatasetFull_Augmented']), (
    f"❌ YOLODatasetFull_Augmented.zip not found in {DRIVE_FOLDER}!\n"
    f"   Expected: {ZIPS['YOLODatasetFull_Augmented']}"
)
assert os.path.exists(ZIPS['HandPD_Spiral']), (
    f"❌ HandPD_Spiral.zip not found in {DRIVE_FOLDER}!\n"
    f"   Expected: {ZIPS['HandPD_Spiral']}"
)

for name, zip_path in ZIPS.items():
    extract_if_needed(zip_path, DATA_DIR, name)

# ═══════════════════════════════════════════════════════════
# Auto-detect extracted folder paths
# ═══════════════════════════════════════════════════════════
def find_folder(base, candidates):
    """Find the first existing folder from a list of candidates."""
    for c in candidates:
        path = os.path.join(base, c)
        if os.path.isdir(path):
            return path
        for sub in (os.listdir(base) if os.path.isdir(base) else []):
            path2 = os.path.join(base, sub, c)
            if os.path.isdir(path2):
                return path2
    return None

YOLO_ROOT = find_folder(DATA_DIR, ['YOLODatasetFull'])
HANDPD_ROOT = find_folder(DATA_DIR, ['HandPD_Augmented_Data', 'HandPD_Spiral'])
UCI_ROOT = find_folder(DATA_DIR, ['UCI', 'UCI_PD_Spiral', 'hw_dataset'])

# Handle double-nested YOLO folder
if YOLO_ROOT and os.path.isdir(os.path.join(YOLO_ROOT, 'YOLODatasetFull')):
    YOLO_ROOT = os.path.join(YOLO_ROOT, 'YOLODatasetFull')

print(f"\n   YOLO root:   {YOLO_ROOT}")
print(f"   HandPD root: {HANDPD_ROOT}")
print(f"   UCI root:    {UCI_ROOT}")

# ═══════════════════════════════════════════════════════════
# UCI: RENDER TABLET CSV DATA → SPIRAL IMAGES
# ═══════════════════════════════════════════════════════════
# The UCI dataset contains X,Y,Z,Pressure,GripAngle,Timestamp,TestID
# from a Wacom digitizer tablet. We render these coordinates into
# 224×224 images — this gives us CLINICAL-GRADE spiral drawings from
# Cerrahpasa Faculty of Medicine (62 PWP + 15 healthy controls).
# ═══════════════════════════════════════════════════════════

def render_tablet_to_image(csv_path, img_size=224, line_width=2, filter_test_id=0):
    """
    Render UCI tablet CSV (X;Y;Z;Pressure;GripAngle;Timestamp;TestID)
    into a PIL image. Uses pressure for line thickness variation.
    """
    try:
        with open(csv_path, 'r') as f:
            lines = f.readlines()
        
        points = []
        for line in lines:
            parts = line.strip().split(';')
            if len(parts) >= 7:
                try:
                    x, y = float(parts[0]), float(parts[1])
                    pressure = float(parts[3])
                    test_id = int(parts[6])
                    if filter_test_id is not None and test_id != filter_test_id:
                        continue
                    points.append((x, y, pressure))
                except (ValueError, IndexError):
                    continue
        
        if len(points) < 10:
            return None
        
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        pressures = [p[2] for p in points]
        
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        x_range = max(x_max - x_min, 1)


In [ ]:
# ============================================================
# CELL 2: Custom Dataset & Advanced Augmentation Pipeline
# ============================================================
# Augmentation strategy (4 levels):
#   Level 1: 12 geometric/photometric transforms (online, per-batch)
#   Level 2: MixUp (α=0.4) — blend two images for smoother decision boundaries
#   Level 3: CutMix (α=1.0) — paste rectangular region for spatial regularization
#   Level 4: DDPM Diffusion (next cell) — generate entirely new images
# ============================================================

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ═══════════════════════════════════════════════════════════
# HEAVY augmentation — 12 transforms for diverse training
# ═══════════════════════════════════════════════════════════
train_transforms = T.Compose([
    T.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    T.RandomCrop(IMG_SIZE),
    T.RandomRotation(30),                                  # ±30° — generous rotation
    T.RandomHorizontalFlip(p=0.5),                         # OK for spirals/waves (symmetric)
    T.RandomVerticalFlip(p=0.3),                           # Also OK
    T.RandomPerspective(distortion_scale=0.15, p=0.4),     # Camera/scanner angle
    T.RandomAffine(
        degrees=0,
        translate=(0.08, 0.08),                            # Position shift
        scale=(0.85, 1.15),                                # Scale variation
        shear=(-10, 10),                                   # Shear deformation
    ),
    T.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.2,
        hue=0.05,
    ),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),      # Pen/scan blur
    T.RandomGrayscale(p=0.3),                              # Many originals are grayscale
    T.RandomInvert(p=0.1),                                 # Some scans have inverted colors
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    T.RandomErasing(p=0.2, scale=(0.02, 0.15)),           # Random occlusions
])

val_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ═══════════════════════════════════════════════════════════
# MixUp & CutMix — Advanced Batch-Level Augmentation
# ═══════════════════════════════════════════════════════════
# MixUp (Zhang et al., 2018): Blend images → smoother boundaries
# CutMix (Yun et al., 2019): Paste patches → spatial regularization
# These are applied during training AFTER DataLoader batching
# ═══════════════════════════════════════════════════════════

def mixup_data(x, y, alpha=0.4):
    """
    MixUp: x_mixed = λ*x_i + (1-λ)*x_j, y_mixed = λ*y_i + (1-λ)*y_j
    Encourages linear behavior between training examples.
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def cutmix_data(x, y, alpha=1.0):
    """
    CutMix: Cut rectangular region from one image and paste onto another.
    Better than Cutout — the cut region contains useful information.
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    # Generate random bounding box
    W, H = x.size(3), x.size(2)
    cut_ratio = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_ratio)
    cut_h = int(H * cut_ratio)
    
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    
    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    
    # Adjust lambda to actual paste area
    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))
    
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Mixed loss for MixUp/CutMix."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# Configuration: probability of applying each during training
MIXUP_PROB = 0.3      # 30% of batches get MixUp
CUTMIX_PROB = 0.2     # 20% of batches get CutMix
# Remaining 50% = standard augmentation only

# ═══════════════════════════════════════════════════════════
# Custom Dataset — supports spiral + wave drawings
# from YOLO + HandPD + UCI datasets
# ═══════════════════════════════════════════════════════════
class ParkinsonsDrawingDataset(Dataset):
    """
    Multi-source Parkinson's spiral & wave drawing dataset.
    Sources: YOLODatasetFull + HandPD + UCI Digitized Graphics Tablet
    Labels: 0 = Healthy, 1 = Parkinson's Disease
    """
    
    def __init__(self, dataframe, transform=None, oversample_factor=1):
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform
        self.oversample_factor = oversample_factor
    
    def __len__(self):
        return len(self.data) * self.oversample_factor
    
    def __getitem__(self, idx):
        real_idx = idx % len(self.data)
        row = self.data.iloc[real_idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, row['label']

CLASS_NAMES = {0: 'Healthy', 1: "Parkinson's"}
print(f"✅ Dataset and augmentation pipeline defined")
print(f"   Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"   Level 1: 12 geometric/photometric transforms")
print(f"   Level 2: MixUp (α=0.4, p={MIXUP_PROB})")
print(f"   Level 3: CutMix (α=1.0, p={CUTMIX_PROB})")
print(f"   Level 4: DDPM Diffusion (next cell)")

## 🔬 Diffusion-Based Synthetic Data Generation (DDPM)

### Why Diffusion Models?
Traditional augmentations (rotation, flip, color jitter) only *transform* existing images. A **Denoising Diffusion Probabilistic Model (DDPM)** learns the *underlying distribution* of drawing patterns and generates **entirely new, realistic samples** — capturing subtle tremor patterns, pressure variations, and stroke morphologies that geometric transforms cannot synthesize.

### Benefits for Motor Assessment:
1. **Class Balancing** — Generate synthetic minority-class (healthy) images to fix imbalance
2. **Distribution Coverage** — Explore unseen regions of the drawing manifold
3. **Regularization** — Diverse synthetic data reduces overfitting on limited real samples
4. **Clinical Robustness** — Model sees more variation in tremor/micrographia patterns

### Implementation:
- **Architecture**: Lightweight U-Net with time embedding + class conditioning
- **Resolution**: 64×64 generation → bicubic upscale to 224×224
- **Schedule**: Cosine β noise schedule (500 timesteps)
- **Training**: ~100 epochs on Colab T4 GPU (~15-20 min)
- **Generation**: Balanced to equalize class distribution

### References:
- Ho et al. (2020) — *"Denoising Diffusion Probabilistic Models"* NeurIPS
- Nichol & Dhariwal (2021) — *"Improved Denoising Diffusion Probabilistic Models"* ICML
- Dhariwal & Nichol (2021) — *"Diffusion Models Beat GANs on Image Synthesis"* NeurIPS

In [ ]:
# ============================================================
# CELL 2B: DDPM — Lightweight Conditional Diffusion Model
# ============================================================
# Generates synthetic spiral/wave drawings to:
#   1. Balance minority class (healthy)
#   2. Increase dataset diversity beyond traditional augments
# ============================================================

import math

# ═══════════════════════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════════════════════
DDPM_IMG_SIZE = 64          # Generate at 64×64, upscale later
DDPM_TIMESTEPS = 500        # Diffusion steps
DDPM_EPOCHS = 100           # Training epochs (~15 min on T4)
DDPM_BATCH_SIZE = 64
DDPM_LR = 2e-4
NUM_CLASSES = 2             # 0=Healthy, 1=PD
USE_DIFFUSION = True        # Set False to skip (uses only traditional augments)

# ═══════════════════════════════════════════════════════════
# Cosine Noise Schedule (Nichol & Dhariwal, 2021)
# Better than linear — preserves more signal at high timesteps
# ═══════════════════════════════════════════════════════════
def cosine_beta_schedule(timesteps, s=0.008):
    """Cosine schedule as proposed in 'Improved DDPM'."""
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)

# Precompute schedule tensors
betas = cosine_beta_schedule(DDPM_TIMESTEPS)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

def extract(a, t, x_shape):
    """Extract values from schedule tensor at timestep t."""
    batch_size = t.shape[0]
    out = a.gather(-1, t.cpu())
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1))).to(t.device)

# ═══════════════════════════════════════════════════════════
# Forward Diffusion: q(x_t | x_0)
# ═══════════════════════════════════════════════════════════
def q_sample(x_start, t, noise=None):
    """Add noise to image at timestep t."""
    if noise is None:
        noise = torch.randn_like(x_start)
    sqrt_alpha = extract(sqrt_alphas_cumprod, t, x_start.shape)
    sqrt_one_minus_alpha = extract(sqrt_one_minus_alphas_cumprod, t, x_start.shape)
    return sqrt_alpha * x_start + sqrt_one_minus_alpha * noise

# ═══════════════════════════════════════════════════════════
# Sinusoidal Time Embedding
# ═══════════════════════════════════════════════════════════
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

# ═══════════════════════════════════════════════════════════
# Residual Block with Time + Class Conditioning
# ═══════════════════════════════════════════════════════════
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_classes=2):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bn1 = nn.GroupNorm(8, out_ch)
        self.bn2 = nn.GroupNorm(8, out_ch)
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_ch)
        )
        self.class_emb = nn.Embedding(num_classes, out_ch)
        self.residual = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb, class_label):
        h = self.bn1(F.silu(self.conv1(x)))
        # Add time embedding
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        # Add class conditioning
        h = h + self.class_emb(class_label)[:, :, None, None]
        h = self.bn2(F.silu(self.conv2(h)))
        return h + self.residual(x)

# ═══════════════════════════════════════════════════════════
# Lightweight U-Net for 64×64 Diffusion
# ═══════════════════════════════════════════════════════════
class LightweightUNet(nn.Module):
    """
    Compact U-Net for class-conditional DDPM.
    Designed for Colab T4 GPU (~1.5GB VRAM).
    
    Architecture:
        Encoder: 3→64→128→256 (with downsampling)
        Bottleneck: 256→256
        Decoder: 256→128→64→3 (with upsampling + skip connections)
        + Sinusoidal time embedding + Class embedding at every block
    """
    def __init__(self, in_channels=3, base_ch=64, time_dim=256, num_classes=2):
        super().__init__()
        
        # Time embedding network
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        
        # Encoder
        self.enc1 = ResBlock(in_channels, base_ch, time_dim, num_classes)
        self.enc2 = ResBlock(base_ch, base_ch * 2, time_dim, num_classes)
        self.enc3 = ResBlock(base_ch * 2, base_ch * 4, time_dim, num_classes)
        
        self.down1 = nn.Conv2d(base_ch, base_ch, 4, stride=2, padding=1)
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 2, 4, stride=2, padding=1)
        self.down3 = nn.Conv2d(base_ch * 4, base_ch * 4, 4, stride=2, padding=1)
        
        # Bottleneck
        self.bottleneck = ResBlock(base_ch * 4, base_ch * 4, time_dim, num_classes)
        
        # Decoder (with skip connections)
        self.up3 = nn.ConvTranspose2d(base_ch * 4, base_ch * 4, 4, stride=2, padding=1)
        self.dec3 = ResBlock(base_ch * 8, base_ch * 2, time_dim, num_classes)  # skip concat
        
        self.up2 = nn.ConvTranspose2d(base_ch * 2, base_ch * 2, 4, stride=2, padding=1)
        self.dec2 = ResBlock(base_ch * 4, base_ch, time_dim, num_classes)       # skip concat
        
        self.up1 = nn.ConvTranspose2d(base_ch, base_ch, 4, stride=2, padding=1)
        self.dec1 = ResBlock(base_ch * 2, base_ch, time_dim, num_classes)       # skip concat
        
        # Output
        self.out_conv = nn.Conv2d(base_ch, in_channels, 1)
    
    def forward(self, x, timestep, class_label):
        t = self.time_mlp(timestep)
        
        # Encoder
        e1 = self.enc1(x, t, class_label)
        e2 = self.enc2(self.down1(e1), t, class_label)
        e3 = self.enc3(self.down2(e2), t, class_label)
        
        # Bottleneck
        b = self.bottleneck(self.down3(e3), t, class_label)
        
        # Decoder with skip connections
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1), t, class_label)
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1), t, class_label)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1), t, class_label)
        
        return self.out_conv(d1)

# ═══════════════════════════════════════════════════════════
# DDPM Sampling (Reverse Diffusion)
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def p_sample(model, x, t, t_index, class_label):
    """Single reverse diffusion step."""
    betas_t = extract(betas, t, x.shape)
    sqrt_one_minus_alpha_t = extract(sqrt_one_minus_alphas_cumprod, t, x.shape)
    sqrt_recip_alpha_t = extract(sqrt_recip_alphas, t, x.shape)
    
    # Predict noise and compute mean
    predicted_noise = model(x, t, class_label)
    model_mean = sqrt_recip_alpha_t * (x - betas_t * predicted_noise / sqrt_one_minus_alpha_t)
    
    if t_index == 0:
        return model_mean
    else:
        posterior_var_t = extract(posterior_variance, t, x.shape)
        noise = torch.randn_like(x)
        return model_mean + torch.sqrt(posterior_var_t) * noise

@torch.no_grad()
def p_sample_loop(model, shape, class_label, device):
    """Full reverse diffusion: noise → image."""
    img = torch.randn(shape, device=device)
    for i in reversed(range(DDPM_TIMESTEPS)):
        t = torch.full((shape[0],), i, device=device, dtype=torch.long)
        img = p_sample(model, img, t, i, class_label)
    return img

@torch.no_grad()
def generate_samples(model, num_samples, target_class, device, batch_size=32):
    """Generate num_samples images of a specific class."""
    model.eval()
    all_images = []
    
    for i in range(0, num_samples, batch_size):
        bs = min(batch_size, num_samples - i)
        class_labels = torch.full((bs,), target_class, device=device, dtype=torch.long)
        images = p_sample_loop(model, (bs, 3, DDPM_IMG_SIZE, DDPM_IMG_SIZE), class_labels, device)
        # Clamp to [-1, 1] then rescale to [0, 1]
        images = torch.clamp(images, -1, 1) * 0.5 + 0.5
        all_images.append(images.cpu())
        print(f"   Generated {min(i + bs, num_samples)}/{num_samples}", end='\r')
    
    return torch.cat(all_images, dim=0)

print("✅ DDPM architecture defined")
params = sum(p.numel() for p in LightweightUNet().parameters())
print(f"   U-Net parameters: {params:,} ({params/1e6:.1f}M)")
print(f"   Diffusion steps: {DDPM_TIMESTEPS}")
print(f"   Generation resolution: {DDPM_IMG_SIZE}×{DDPM_IMG_SIZE}")
print(f"   Noise schedule: cosine (Nichol & Dhariwal, 2021)")

In [ ]:
# ============================================================
# CELL 2C: Train DDPM & Generate Synthetic Spiral/Wave Data
# ============================================================
# This cell:
#   1. Prepares a 64×64 dataset from real images
#   2. Trains the conditional DDPM (~15-20 min on T4)
#   3. Generates synthetic minority-class images
#   4. Saves them to disk and adds to df_all
#   5. Visualizes real vs synthetic comparison
# ============================================================

if USE_DIFFUSION:
    print("=" * 70)
    print("🔬 DDPM SYNTHETIC DATA GENERATION")
    print("=" * 70)
    
    # ─── Prepare 64×64 dataset for DDPM training ───
    ddpm_transforms = T.Compose([
        T.Resize((DDPM_IMG_SIZE, DDPM_IMG_SIZE)),
        T.RandomHorizontalFlip(p=0.5),
        T.ToTensor(),
        T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # Scale to [-1, 1]
    ])
    
    class DDPMDataset(Dataset):
        """Simple dataset for DDPM training at 64×64."""
        def __init__(self, dataframe, transform):
            self.data = dataframe.reset_index(drop=True)
            self.transform = transform
        def __len__(self):
            return len(self.data)
        def __getitem__(self, idx):
            row = self.data.iloc[idx]
            img = Image.open(row['path']).convert('RGB')
            return self.transform(img), row['label']
    
    ddpm_dataset = DDPMDataset(df_all, ddpm_transforms)
    ddpm_loader = DataLoader(
        ddpm_dataset, batch_size=DDPM_BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True
    )
    
    # ─── Class distribution analysis ───
    class_dist = df_all['label'].value_counts().sort_index()
    minority_class = class_dist.idxmin()
    majority_class = class_dist.idxmax()
    imbalance_gap = class_dist[majority_class] - class_dist[minority_class]
    
    print(f"\n📊 Current class distribution:")
    print(f"   Healthy (0): {class_dist.get(0, 0)} images")
    print(f"   PD (1):      {class_dist.get(1, 0)} images")
    print(f"   Minority class: {CLASS_NAMES[minority_class]} (label={minority_class})")
    print(f"   Imbalance gap: {imbalance_gap} images")
    
    # Generate enough to balance + add 20% extra diversity
    n_synthetic = int(imbalance_gap * 1.2) if imbalance_gap > 100 else 500
    print(f"   Will generate: {n_synthetic} synthetic {CLASS_NAMES[minority_class]} images")
    
    # ─── Initialize model ───
    ddpm_model = LightweightUNet(
        in_channels=3, base_ch=64, time_dim=256, num_classes=NUM_CLASSES
    ).to(device)
    
    ddpm_optimizer = torch.optim.AdamW(ddpm_model.parameters(), lr=DDPM_LR, weight_decay=1e-4)
    ddpm_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ddpm_optimizer, T_max=DDPM_EPOCHS)
    
    # ─── Training Loop ───
    print(f"\n🏋️ Training DDPM for {DDPM_EPOCHS} epochs...")
    print(f"   Batches/epoch: {len(ddpm_loader)}")
    
    ddpm_losses = []
    best_ddpm_loss = float('inf')
    
    for epoch in range(DDPM_EPOCHS):
        ddpm_model.train()
        epoch_loss = 0.0
        
        for batch_imgs, batch_labels in ddpm_loader:
            batch_imgs = batch_imgs.to(device)
            batch_labels = batch_labels.to(device).long()
            
            # Sample random timesteps
            t = torch.randint(0, DDPM_TIMESTEPS, (batch_imgs.shape[0],), device=device).long()
            
            # Add noise (forward diffusion)
            noise = torch.randn_like(batch_imgs)
            x_noisy = q_sample(batch_imgs, t, noise)
            
            # Predict noise
            predicted_noise = ddpm_model(x_noisy, t, batch_labels)
            
            # Simple MSE loss (Ho et al., 2020)
            loss = F.mse_loss(predicted_noise, noise)
            
            ddpm_optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ddpm_model.parameters(), 1.0)
            ddpm_optimizer.step()
            
            epoch_loss += loss.item()
        
        ddpm_scheduler.step()
        avg_loss = epoch_loss / len(ddpm_loader)
        ddpm_losses.append(avg_loss)
        
        if avg_loss < best_ddpm_loss:
            best_ddpm_loss = avg_loss
            torch.save(ddpm_model.state_dict(), 'ddpm_spiral_best.pt')
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"   Epoch {epoch+1:3d}/{DDPM_EPOCHS} | Loss: {avg_loss:.4f} | Best: {best_ddpm_loss:.4f} | LR: {ddpm_scheduler.get_last_lr()[0]:.2e}")
    
    print(f"\n✅ DDPM training complete! Best loss: {best_ddpm_loss:.4f}")
    
    # ─── Plot DDPM training loss ───
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    ax.plot(ddpm_losses, color='purple', linewidth=1.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('DDPM Training Loss (Noise Prediction)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # ─── Load best model and generate synthetic images ───
    ddpm_model.load_state_dict(torch.load('ddpm_spiral_best.pt'))
    
    print(f"\n🎨 Generating {n_synthetic} synthetic {CLASS_NAMES[minority_class]} images...")
    synthetic_images = generate_samples(ddpm_model, n_synthetic, minority_class, device, batch_size=32)
    
    # ─── Save synthetic images to disk ───
    synthetic_dir = os.path.join(BASE_DIR, 'synthetic_spiral')
    os.makedirs(synthetic_dir, exist_ok=True)
    
    synthetic_records = []
    upscale_transform = T.Resize((224, 224), interpolation=T.InterpolationMode.BICUBIC, antialias=True)
    
    for i, img_tensor in enumerate(synthetic_images):
        # Upscale from 64×64 to 224×224
        img_upscaled = upscale_transform(img_tensor)
        img_pil = T.ToPILImage()(img_upscaled)
        
        save_path = os.path.join(synthetic_dir, f'synthetic_{CLASS_NAMES[minority_class].lower()}_{i:04d}.png')
        img_pil.save(save_path, quality=95)
        
        synthetic_records.append({
            'path': save_path,
            'label': minority_class,
            'class_name': CLASS_NAMES[minority_class],
            'drawing_type': 'spiral',  # Primary type
            'dataset': 'DDPM_Synthetic',
            'split': 'train',
            'source': 'ddpm_generated'
        })
    
    df_synthetic = pd.DataFrame(synthetic_records)
    
    # ─── Add synthetic data to df_all ───
    df_all_original_size = len(df_all)
    df_all = pd.concat([df_all, df_synthetic], ignore_index=True)
    
    print(f"\n📊 Dataset after DDPM augmentation:")
    print(f"   Original: {df_all_original_size} images")
    print(f"   Synthetic: +{len(df_synthetic)} {CLASS_NAMES[minority_class]} images")
    print(f"   Total:    {len(df_all)} images")
    
    new_dist = df_all['label'].value_counts().sort_index()
    print(f"\n   New distribution:")
    print(f"   Healthy (0): {new_dist.get(0, 0)} images")
    print(f"   PD (1):      {new_dist.get(1, 0)} images")
    ratio = new_dist.min() / new_dist.max()
    print(f"   Balance ratio: {ratio:.2f} (1.0 = perfect)")
    
    # ─── Visualize: Real vs Synthetic comparison ───
    fig, axes = plt.subplots(3, 8, figsize=(20, 8))
    fig.suptitle('DDPM Synthetic Data Generation — Real vs Synthetic Comparison', fontsize=14, fontweight='bold')
    
    # Row 1: Real healthy samples
    real_healthy = df_all[df_all['label'] == 0].sample(min(8, len(df_all[df_all['label'] == 0])))
    for i, (_, row) in enumerate(real_healthy.head(8).iterrows()):
        img = Image.open(row['path']).convert('RGB').resize((128, 128))
        axes[0, i].imshow(np.array(img))
        axes[0, i].set_title('Real Healthy', fontsize=8, color='green')
        axes[0, i].axis('off')
    axes[0, 0].set_ylabel('Real\nHealthy', fontsize=10, fontweight='bold')
    
    # Row 2: Synthetic healthy samples
    synth_paths = df_synthetic['path'].tolist()
    for i in range(min(8, len(synth_paths))):
        img = Image.open(synth_paths[i]).convert('RGB').resize((128, 128))
        axes[1, i].imshow(np.array(img))
        axes[1, i].set_title('Synthetic', fontsize=8, color='blue')
        axes[1, i].axis('off')
    axes[1, 0].set_ylabel('DDPM\nSynthetic', fontsize=10, fontweight='bold')
    
    # Row 3: Real PD samples
    real_pd = df_all[df_all['label'] == 1].sample(min(8, len(df_all[df_all['label'] == 1])))
    for i, (_, row) in enumerate(real_pd.head(8).iterrows()):
        img = Image.open(row['path']).convert('RGB').resize((128, 128))
        axes[2, i].imshow(np.array(img))
        axes[2, i].set_title('Real PD', fontsize=8, color='red')
        axes[2, i].axis('off')
    axes[2, 0].set_ylabel('Real\nPD', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # ─── Quality check: random synthetic grid ───
    fig, axes = plt.subplots(4, 8, figsize=(20, 10))
    fig.suptitle(f'Generated {CLASS_NAMES[minority_class]} Samples (DDPM, {DDPM_TIMESTEPS} steps)', fontsize=14, fontweight='bold')
    for i in range(32):
        r, c = i // 8, i % 8
        if i < len(synthetic_images):
            img = synthetic_images[i].permute(1, 2, 0).numpy()
            img = np.clip(img, 0, 1)
            axes[r, c].imshow(img)
        axes[r, c].axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"\n✅ Synthetic data saved to: {synthetic_dir}")
    print(f"   {len(synthetic_records)} images ready for training")

else:
    print("⏭️ Diffusion generation SKIPPED (USE_DIFFUSION = False)")
    print("   Using only traditional augmentations")

In [ ]:
# ============================================================
# CELL 3: PATIENT-LEVEL Train/Val/Test Split & DataLoaders
# ============================================================
# ⚠️  CRITICAL FIX: Patient-level splitting prevents data leakage
#
# HandPD contains ~92 subjects drawing multiple spirals/waves.
# The original pre-defined train/test folders split at IMAGE level,
# meaning the same patient can appear in both train and test.
# This inflates accuracy because the model memorizes patient-specific
# pen stroke signatures rather than learning PD motor patterns.
#
# FIX: Extract patient IDs from HandPD filenames and ensure ALL
# images from one patient go to the SAME split.
# ============================================================

import re
from sklearn.model_selection import GroupShuffleSplit

# ═══════════════════════════════════════════════════════════
# STEP 1: Extract patient IDs from filenames
# ═══════════════════════════════════════════════════════════
# HandPD filename patterns:
#   Original:  "V01_1_spiral.png"  → patient = "handpd_V01"
#   Augmented: "aug_V01_1_spiral.png" → patient = "handpd_V01"
#   Variations: "V02_01.png", "v03_2_meander.png"
# UCI:  "control_001.txt" → patient = "uci_001"
# YOLO: "healthy_spiral_0042.png" → no patient ID available
#       → YOLO images are crowd-sourced/augmented, treat each as independent

def extract_patient_id(row):
    """
    Extract a patient-level identifier from the file path.
    Returns a string like 'handpd_V01', 'uci_003', or 'yolo_<hash>' for
    images without patient metadata.
    """
    fname = os.path.basename(row['path']).lower()
    dataset = row.get('dataset', '').lower()

    # ── HandPD: extract volunteer number (V01, V02, ...) ──
    if 'handpd' in dataset:
        # Match V followed by digits, possibly prefixed by 'aug_'
        match = re.search(r'v(\d+)', fname)
        if match:
            return f"handpd_V{match.group(1).zfill(2)}"
        # Fallback: try first number group as patient ID
        match = re.search(r'(\d+)', fname)
        if match:
            return f"handpd_{match.group(1).zfill(3)}"

    # ── UCI: patient number from folder or filename ──
    if 'uci' in dataset:
        # UCI files are in folders like control/V01_1.txt or parkinson/V01_1.txt
        parent = os.path.basename(os.path.dirname(row['path'])).lower()
        match = re.search(r'v(\d+)', fname)
        if match:
            return f"uci_V{match.group(1).zfill(2)}"
        match = re.search(r'(\d+)', fname)
        if match:
            return f"uci_{match.group(1).zfill(3)}"

    # ── YOLO: no patient metadata — each image is independent ──
    # Use filename hash so each image gets a unique "patient"
    # This means YOLO images can freely split (no grouping constraint)
    return f"yolo_{hashlib.md5(fname.encode()).hexdigest()[:8]}"

# Add patient_id column
df_all['patient_id'] = df_all.apply(extract_patient_id, axis=1)

# ═══════════════════════════════════════════════════════════
# STEP 2: Verify patient extraction
# ═══════════════════════════════════════════════════════════
n_patients = df_all['patient_id'].nunique()
n_handpd = df_all[df_all['patient_id'].str.startswith('handpd')]['patient_id'].nunique()
n_uci = df_all[df_all['patient_id'].str.startswith('uci')]['patient_id'].nunique()
n_yolo = df_all[df_all['patient_id'].str.startswith('yolo')]['patient_id'].nunique()

print("=" * 70)
print("🔒 PATIENT-LEVEL SPLIT (prevents data leakage)")
print("=" * 70)
print(f"\n   Patient IDs extracted:")
print(f"     HandPD subjects: {n_handpd}")
print(f"     UCI subjects:    {n_uci}")
print(f"     YOLO (indep.):   {n_yolo} unique images")
print(f"     Total groups:    {n_patients}")

# Images per patient (HandPD should have multiple)
handpd_mask = df_all['patient_id'].str.startswith('handpd')
if handpd_mask.sum() > 0:
    imgs_per_patient = df_all[handpd_mask].groupby('patient_id').size()
    print(f"\n   HandPD images/subject: mean={imgs_per_patient.mean():.1f}, "
          f"min={imgs_per_patient.min()}, max={imgs_per_patient.max()}")

# ═══════════════════════════════════════════════════════════
# STEP 3: Patient-level split using GroupShuffleSplit
# ═══════════════════════════════════════════════════════════
# ALL images from one patient → same split (train, val, or test)
# Stratify by class to maintain class balance across splits

patients = df_all.groupby('patient_id').agg({
    'class_name': 'first',   # patient's diagnosis
    'label': 'first',
}).reset_index()

# First split: 85% trainval / 15% test (at patient level)
from sklearn.model_selection import train_test_split

pat_trainval, pat_test = train_test_split(
    patients, test_size=0.15, random_state=SEED,
    stratify=patients['label']
)

# Second split: ~82% train / ~18% val of trainval ≈ 70/15 overall
pat_train, pat_val = train_test_split(
    pat_trainval, test_size=0.176, random_state=SEED,
    stratify=pat_trainval['label']
)

# Map back to images
train_pids = set(pat_train['patient_id'])
val_pids = set(pat_val['patient_id'])
test_pids = set(pat_test['patient_id'])

df_train = df_all[df_all['patient_id'].isin(train_pids)].copy()
df_val = df_all[df_all['patient_id'].isin(val_pids)].copy()
df_test_orig = df_all[df_all['patient_id'].isin(test_pids)].copy()

# ═══════════════════════════════════════════════════════════
# STEP 4: Verify NO patient leakage
# ═══════════════════════════════════════════════════════════
train_patients = set(df_train['patient_id'])
val_patients = set(df_val['patient_id'])
test_patients = set(df_test_orig['patient_id'])

leak_tv = train_patients & val_patients
leak_tt = train_patients & test_patients
leak_vt = val_patients & test_patients

assert len(leak_tv) == 0, f"❌ TRAIN-VAL LEAKAGE: {leak_tv}"
assert len(leak_tt) == 0, f"❌ TRAIN-TEST LEAKAGE: {leak_tt}"
assert len(leak_vt) == 0, f"❌ VAL-TEST LEAKAGE: {leak_vt}"
print(f"\n   ✅ ZERO patient leakage verified!")
print(f"      Train patients: {len(train_patients)}")
print(f"      Val patients:   {len(val_patients)}")
print(f"      Test patients:  {len(test_patients)}")
print(f"      Overlap: NONE ✓")

# HandPD-specific leakage check
handpd_train = {p for p in train_patients if p.startswith('handpd')}
handpd_test = {p for p in test_patients if p.startswith('handpd')}
handpd_val = {p for p in val_patients if p.startswith('handpd')}
print(f"\n   🔒 HandPD subject isolation:")
print(f"      Train: {len(handpd_train)} subjects → {df_train[df_train['patient_id'].str.startswith('handpd')].shape[0]} images")
print(f"      Val:   {len(handpd_val)} subjects → {df_val[df_val['patient_id'].str.startswith('handpd')].shape[0]} images")
print(f"      Test:  {len(handpd_test)} subjects → {df_test_orig[df_test_orig['patient_id'].str.startswith('handpd')].shape[0]} images")

print(f"\n{'='*70}")
print("📊 SPLIT SUMMARY (PATIENT-LEVEL)")
print("=" * 70)
print(f"   Train: {len(df_train)} images ({len(train_patients)} patients)")
print(f"   Val:   {len(df_val)} images ({len(val_patients)} patients)")
print(f"   Test:  {len(df_test_orig)} images ({len(test_patients)} patients)")
print(f"   Total: {len(df_train) + len(df_val) + len(df_test_orig)} images")

for split_name, df_split in [("Train", df_train), ("Val", df_val), ("Test", df_test_orig)]:
    print(f"\n   {split_name} breakdown:")
    for _, row in df_split.groupby(['drawing_type', 'class_name']).size().reset_index(name='n').iterrows():
        print(f"     {row['drawing_type']:8s} | {row['class_name']:10s} | {row['n']}")
    print(f"     Sources: {', '.join(df_split['dataset'].unique())}")

# ═══════════════════════════════════════════════════════════
# Weighted Sampler for balanced training
# ═══════════════════════════════════════════════════════════
class_counts = df_train['label'].value_counts().sort_index()
class_weights = 1.0 / class_counts.values
sample_weights = df_train['label'].map(dict(zip(class_counts.index, class_weights))).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

healthy_count = int(class_counts.values[0])
pd_count      = int(class_counts.values[1])
balance_ratio = max(healthy_count, pd_count) / min(healthy_count, pd_count)
print(f"\n   Class counts  : Healthy={healthy_count:,}  PD={pd_count:,}")
print(f"   Class weights : Healthy=1/{healthy_count:,} ({class_weights[0]:.2e})  PD=1/{pd_count:,} ({class_weights[1]:.2e})")
print(f"   Class balance : {balance_ratio:.3f}x  ({'✅ well-balanced' if balance_ratio < 1.2 else '⚠️ imbalanced — sampler active'})")

# ═══════════════════════════════════════════════════════════
# Oversampling factor — adaptive based on dataset size
# ═══════════════════════════════════════════════════════════
n_train = len(df_train)
if n_train < 500:
    OVERSAMPLE = 5
elif n_train < 2000:
    OVERSAMPLE = 2
else:
    OVERSAMPLE = 1

BATCH_SIZE = 64 if n_train >= 2000 else (32 if n_train >= 300 else 16)

train_dataset = ParkinsonsDrawingDataset(df_train, transform=train_transforms, oversample_factor=OVERSAMPLE)
val_dataset = ParkinsonsDrawingDataset(df_val, transform=val_transforms)
test_dataset = ParkinsonsDrawingDataset(df_test_orig, transform=val_transforms)

oversample_weights = np.tile(sample_weights, OVERSAMPLE)
oversample_sampler = WeightedRandomSampler(
    weights=oversample_weights,
    num_samples=len(oversample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=oversample_sampler,
    num_workers=2, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"\n   Dataset size: {n_train} → Oversampling: {OVERSAMPLE}x → {len(train_dataset)} effective samples")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Train batches/epoch: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

# ═══════════════════════════════════════════════════════════
# Preview augmented samples
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Augmented Training Samples (different augmentations)', fontsize=14, fontweight='bold')

for i in range(12):
    row, col = i // 6, i % 6
    img, label = train_dataset[i]
    img_show = img.permute(1, 2, 0).numpy()
    img_show = img_show * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_show = np.clip(img_show, 0, 1)
    axes[row, col].imshow(img_show)
    axes[row, col].set_title(CLASS_NAMES[label], fontsize=9,
                              color='green' if label == 0 else 'red')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()
print(f"✅ DataLoaders ready — PATIENT-LEVEL split, {OVERSAMPLE}x oversampling, batch size {BATCH_SIZE}")

## 5️⃣ Model Architecture — MotorNet (EfficientNet-B0 + PD Head)

### Why EfficientNet-B0 for small datasets?
- **ImageNet pretraining** transfers low-level features (edges, textures, curves) directly useful for tremor detection
- **5.3M parameters** but only ~800K trainable after freezing — reduces overfitting risk
- **Compound scaling** — optimal depth/width/resolution balance
- **GradCAM-ready** — conv_head layer gives clinically interpretable saliency maps

### Architecture:
```
EfficientNet-B0 (freeze blocks 0-4, fine-tune blocks 5-6)
    ↓ 1280-d feature vector
Dropout(0.5) → Linear(1280, 256) → ReLU → BatchNorm → Dropout(0.3)
    ↓
Linear(256, 64) → ReLU → BatchNorm
    ↓
Linear(64, 2) ← Healthy vs PD
    ↓ (parallel branch)
Linear(2, 16) → ReLU → Linear(16, 1) → Sigmoid ← PD risk [0,1]
```

### Higher dropout for small dataset:
- `Dropout(0.5)` at the top layer — aggressive regularization
- Smaller hidden dimensions (256→64 instead of 512→128)

In [ ]:
# ============================================================
# CELL 6: Model Architecture — MotorNet
# ============================================================

class MotorNet(nn.Module):
    """
    Parkinson's Disease motor drawing classifier based on EfficientNet-B0.
    
    Trained on spiral + wave drawings from Zham et al. (2017).
    Outputs:
    - class_logits: [Healthy, PD] binary classification
    - pd_risk: Continuous PD risk score [0, 1]
    
    GradCAM compatible — target_layer = self.backbone.conv_head
    """
    
    def __init__(self, num_classes=2, pretrained=True, dropout=0.5):
        super().__init__()
        
        # ── EfficientNet-B0 backbone ──
        self.backbone = timm.create_model(
            'efficientnet_b0',
            pretrained=pretrained,
            num_classes=0,       # Remove classifier head
            global_pool='avg'    # → 1280-d feature vector
        )
        
        # Freeze bottom layers (blocks 0-4)
        # Fine-tune blocks 5, 6, conv_head, bn2
        for name, param in self.backbone.named_parameters():
            if any(x in name for x in ['blocks.5', 'blocks.6', 'conv_head', 'bn2']):
                param.requires_grad = True
            else:
                param.requires_grad = False
        
        trainable = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        frozen = sum(p.numel() for p in self.backbone.parameters() if not p.requires_grad)
        
        # ── Classification head (smaller for tiny dataset) ──
        feature_dim = self.backbone.num_features  # 1280
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),                    # 0.5 — aggressive for small data
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout * 0.6),              # 0.3
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(64),
            nn.Linear(64, num_classes),
        )
        
        # ── PD Risk head (regression) ──
        self.risk_head = nn.Sequential(
            nn.Linear(num_classes, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
        
        head_params = sum(p.numel() for p in self.classifier.parameters()) + \
                      sum(p.numel() for p in self.risk_head.parameters())
        
        print(f"🏗️  MotorNet Architecture:")
        print(f"   Backbone: EfficientNet-B0 ({frozen:,} frozen + {trainable:,} trainable)")
        print(f"   Head: {head_params:,} parameters")
        print(f"   Total trainable: {trainable + head_params:,}")
        print(f"   Dropout: {dropout} (aggressive for small dataset)")
        print(f"   Output: {num_classes} classes + PD risk score")
    
    def forward(self, x):
        features = self.backbone(x)          # [B, 1280]
        logits = self.classifier(features)   # [B, 2]
        
        # PD risk from class probabilities
        with torch.no_grad():
            probs = torch.softmax(logits, dim=-1)
        risk = self.risk_head(probs)         # [B, 1]
        
        return logits, risk.squeeze(-1)
    
    def get_features(self, x):
        """Get backbone features for GradCAM."""
        return self.backbone(x)


# ═══════════════════════════════════════════════════════════
# Instantiate model
# ═══════════════════════════════════════════════════════════
NUM_CLASSES = 2
model = MotorNet(num_classes=NUM_CLASSES, pretrained=True, dropout=0.5).to(device)

# Verify with dummy input
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
logits, risk = model(dummy)
print(f"\n✅ Forward pass test:")
print(f"   Input:  {dummy.shape}")
print(f"   Logits: {logits.shape} → {CLASS_NAMES}")
print(f"   Risk:   {risk.shape} (PD risk scores)")
print(f"   Risk values: {risk.detach().cpu().numpy()}")

## 6️⃣ Training Configuration

### Strategy for Small Dataset:
- **Label Smoothing (0.15)** — stronger than CDT (0.1) to prevent overconfident predictions
- **Weight Decay (5e-4)** — stronger L2 regularization
- **Cosine Annealing with Warm Restarts** — escape local minima
- **Early Stopping (patience=15)** — generous patience due to small/noisy validation set
- **Gradient clipping** — stability for small batches

In [ ]:

# ============================================================
# CELL 5: Training Loop with Early Stopping
# ============================================================

# ═══════════════════════════════════════════════════════════
# Hyperparameters (tuned for 12K+ image combined dataset)
# ═══════════════════════════════════════════════════════════
NUM_EPOCHS = 80             # Must complete cycle 3 (epochs 31-70) of CosineWarmRestarts
                            # T_0=10, T_mult=2 → cycles: 10, 20, 40, 80 epochs
                            # Cycle 3 ends at epoch 70 — best refinement happens near LR minimum
LEARNING_RATE = 3e-4        # Moderate — enough data for stable gradients
WEIGHT_DECAY = 1e-4         # Moderate L2 regularization
LABEL_SMOOTHING = 0.10      # Mild smoothing (less needed with more data)
PATIENCE = 25               # Must be > half of cycle-3 length (40) to avoid mid-cycle stop

# ═══════════════════════════════════════════════════════════
# Loss, Optimizer, Scheduler
# ═══════════════════════════════════════════════════════════
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

# Differential learning rates
backbone_params = [p for n, p in model.backbone.named_parameters() if p.requires_grad]
head_params = list(model.classifier.parameters()) + list(model.risk_head.parameters())

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': LEARNING_RATE / 10},   # Backbone: very slow
    {'params': head_params, 'lr': LEARNING_RATE},              # Head: normal
], weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-7
)

# Mixed precision
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

# ═══════════════════════════════════════════════════════════
# Training functions
# ═══════════════════════════════════════════════════════════
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    """Training with MixUp/CutMix batch-level augmentation."""
    model.train()
    running_loss = 0
    all_preds, all_labels = [], []
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # ── Batch-level augmentation: MixUp / CutMix / Standard ──
        r = np.random.rand()
        use_mix = False
        
        if r < MIXUP_PROB:
            # MixUp augmentation
            images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=0.4)
            use_mix = True
        elif r < MIXUP_PROB + CUTMIX_PROB:
            # CutMix augmentation
            images, labels_a, labels_b, lam = cutmix_data(images, labels, alpha=1.0)
            use_mix = True
        
        optimizer.zero_grad()
        
        if scaler:
            with torch.amp.autocast('cuda'):
                logits, risk = model(images)
                if use_mix:
                    loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
                else:
                    loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, risk = model(images)
            if use_mix:
                loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
            else:
                loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        # For metrics, use original labels (before mix)
        if use_mix:
            all_labels.extend(labels_a.cpu().numpy())
        else:
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    all_preds, all_labels, all_probs = [], [], []
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        
        logits, risk = model(images)
        loss = criterion(logits, labels)
        
        running_loss += loss.item() * images.size(0)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs)
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1, np.array(all_preds), np.array(all_labels), np.array(all_probs)


# ═══════════════════════════════════════════════════════════
# TRAINING LOOP
# ═══════════════════════════════════════════════════════════
print("=" * 60)
print("🚀 STARTING TRAINING — MotorNet (Spiral + Wave)")
print("=" * 60)
print(f"   Dataset: {len(df_train)} train / {len(df_val)} val / {len(df_test_orig)} test")
print(f"   Sources: YOLO + HandPD + UCI Digitized Tablet + DDPM Synthetic")
print(f"   Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}")
print(f"   Oversample: {OVERSAMPLE}x | Label Smoothing: {LABEL_SMOOTHING}")
print(f"   MixUp: p={MIXUP_PROB} α=0.4 | CutMix: p={CUTMIX_PROB} α=1.0")
print(f"   Early stopping patience: {PATIENCE}")
print(f"   Cosine schedule: T_0=10, T_mult=2  →  cycles: 10, 20, 40 epochs")
print()

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': [],
    'train_f1': [], 'val_f1': [],
    'lr': []
}

best_val_f1 = 0
patience_counter = 0
best_epoch = 0

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device
    )
    
    # Validate
    val_loss, val_acc, val_f1, _, _, _ = validate(
        model, val_loader, criterion, device
    )
    
    # Scheduler step
    scheduler.step()
    current_lr = optimizer.param_groups[1]['lr']
    
    # Log
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    history['lr'].append(current_lr)
    
    # Print
    print(f"  Epoch {epoch+1:3d}/{NUM_EPOCHS} │ "
          f"Train: loss={train_loss:.4f} acc={train_acc:.3f} f1={train_f1:.3f} │ "
          f"Val: loss={val_loss:.4f} acc={val_acc:.3f} f1={val_f1:.3f} │ "
          f"LR: {current_lr:.2e}", end="")
    
    # Save best
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch + 1
        patience_counter = 0
        
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
            'val_acc': val_acc,
            'num_classes': NUM_CLASSES,
            'img_size': IMG_SIZE,
            'class_names': CLASS_NAMES,
        }, os.path.join(MODEL_DIR, 'motor_best.pt'))
        
        print(f" ★ BEST", end="")
    else:
        patience_counter += 1
    
    print()
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\n⏹️  Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
        break

print(f"\n{'='*60}")
print(f"✅ TRAINING COMPLETE!")
print(f"   Best epoch: {best_epoch} with val F1 = {best_val_f1:.4f}")
print(f"{'='*60}")


In [ ]:
# ============================================================
# CELL 5b: Progressive Fine-Tuning (Phase 2)
# ============================================================
# Phase 1 (Cell 5) only trained blocks 5,6 + conv_head + bn2.
# Phase 2 unfreezes blocks 3 & 4 and fine-tunes at a much lower
# learning rate, squeezing the remaining accuracy gain by adapting
# mid-level feature detectors (curvature, texture, pen pressure).
#
# Why this matters for PD spiral drawings:
#   blocks 0-2: edges/gabor (keep frozen — ImageNet features are fine)
#   blocks 3-4: curvature/shape detectors ← unfreeze here
#   blocks 5-6: high-level patterns ← already unfrozen in Phase 1
# ============================================================

FT_EPOCHS    = 25           # Fine-tune epochs
FT_LR        = 5e-5         # Very low — prevent catastrophic forgetting
FT_PATIENCE  = 15           # Generous patience for this stable phase

# ── Load best Phase-1 checkpoint ──
checkpoint = torch.load(os.path.join(MODEL_DIR, 'motor_best.pt'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded Phase-1 checkpoint (epoch {checkpoint['epoch']}, val F1={checkpoint['val_f1']:.4f})")

# ── Progressive unfreeze: blocks 3, 4 (in addition to 5, 6) ──
for name, param in model.backbone.named_parameters():
    if any(x in name for x in ['blocks.3', 'blocks.4', 'blocks.5', 'blocks.6', 'conv_head', 'bn2']):
        param.requires_grad = True

trainable_ft = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Phase 2 trainable params: {trainable_ft:,} (blocks 3-6 + head)")

# ── New optimizer with warm-up cosine schedule ──
ft_backbone_params = [p for n, p in model.backbone.named_parameters() if p.requires_grad]
ft_head_params = list(model.classifier.parameters()) + list(model.risk_head.parameters())

ft_optimizer = optim.AdamW([
    {'params': ft_backbone_params, 'lr': FT_LR / 5},   # Backbone: ultra-slow (1e-5)
    {'params': ft_head_params,     'lr': FT_LR},         # Head: slow (5e-5)
], weight_decay=2e-4)                                    # Slightly stronger regularization

# Cosine decay (no restarts — steady annealing for fine-tuning)
ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    ft_optimizer, T_max=FT_EPOCHS, eta_min=1e-8
)

ft_scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

# ── Fine-tuning loop ──
print(f"\n{'='*60}")
print(f"🔧 PHASE 2: PROGRESSIVE FINE-TUNING")
print(f"{'='*60}")
print(f"   Unfrozen: blocks 3-6 + conv_head + bn2 + classifier")
print(f"   Epochs: {FT_EPOCHS} | Backbone LR: {FT_LR/5:.1e} | Head LR: {FT_LR:.1e}")
print(f"   Starting from val F1 = {checkpoint['val_f1']:.4f}")
print()

ft_history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': [], 'lr': []}
ft_best_f1 = checkpoint['val_f1']   # Must beat Phase-1 best
ft_best_epoch = 0
ft_patience_counter = 0

for epoch in range(FT_EPOCHS):
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, criterion, ft_optimizer, ft_scaler, device
    )
    val_loss, val_acc, val_f1, _, _, _ = validate(
        model, val_loader, criterion, device
    )
    
    ft_scheduler.step()
    current_lr = ft_optimizer.param_groups[1]['lr']
    
    ft_history['train_loss'].append(train_loss)
    ft_history['val_loss'].append(val_loss)
    ft_history['train_f1'].append(train_f1)
    ft_history['val_f1'].append(val_f1)
    ft_history['lr'].append(current_lr)
    
    print(f"  FT Epoch {epoch+1:3d}/{FT_EPOCHS} │ "
          f"Train: loss={train_loss:.4f} acc={train_acc:.3f} f1={train_f1:.3f} │ "
          f"Val: loss={val_loss:.4f} acc={val_acc:.3f} f1={val_f1:.3f} │ "
          f"LR: {current_lr:.2e}", end="")
    
    if val_f1 > ft_best_f1:
        ft_best_f1 = val_f1
        ft_best_epoch = epoch + 1
        ft_patience_counter = 0
        
        torch.save({
            'epoch': checkpoint['epoch'] + epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': ft_optimizer.state_dict(),
            'val_f1': val_f1,
            'val_acc': val_acc,
            'num_classes': NUM_CLASSES,
            'img_size': IMG_SIZE,
            'class_names': CLASS_NAMES,
            'phase': 'fine_tune',
        }, os.path.join(MODEL_DIR, 'motor_best.pt'))
        
        print(f" ★ BEST", end="")
        best_val_f1 = ft_best_f1   # Update global best for curves cell
        best_epoch  = checkpoint['epoch'] + epoch + 1
    else:
        ft_patience_counter += 1
    
    print()
    
    if ft_patience_counter >= FT_PATIENCE:
        print(f"\n⏹️  Fine-tune early stop at FT epoch {epoch+1} (no improvement for {FT_PATIENCE} epochs)")
        break

print(f"\n{'='*60}")
print(f"✅ FINE-TUNING COMPLETE!")
if ft_best_epoch > 0:
    print(f"   Best FT epoch: {ft_best_epoch}/{FT_EPOCHS}  val F1 = {ft_best_f1:.4f}")
    gain = ft_best_f1 - checkpoint['val_f1']
    print(f"   Improvement from Phase 1: +{gain:.4f} ({gain*100:.1f}% points)")
else:
    print(f"   No improvement over Phase 1 (Phase-1 checkpoint kept)")
print(f"{'='*60}")

# Merge ft_history into history for unified curves
history['train_loss'] += ft_history['train_loss']
history['val_loss']   += ft_history['val_loss']
history['train_f1']   += ft_history['train_f1']
history['val_f1']     += ft_history['val_f1']
history['lr']         += ft_history['lr']
# Pad acc arrays so they stay aligned (np.nan = matplotlib-safe gap)
history['train_acc'] += [np.nan] * len(ft_history['train_loss'])
history['val_acc']   += [np.nan] * len(ft_history['val_loss'])
phase1_len = len(history['train_loss']) - len(ft_history['train_loss'])  # epoch index of phase boundary


In [ ]:

# ============================================================
# CELL 8: Training Curves
# ============================================================

# Phase boundary (None if no fine-tuning was run)
phase_boundary = phase1_len if 'phase1_len' in globals() else None

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('MotorNet Training Curves  (Phase 1: cosine restarts | Phase 2: fine-tune)',
             fontsize=13, fontweight='bold')

def _add_phase(ax):
    if phase_boundary:
        ax.axvline(phase_boundary - 0.5, color='orange', linestyle=':', linewidth=2,
                   alpha=0.8, label='Phase 2 start')

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
_add_phase(axes[0])
axes[0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', linewidth=2)
axes[1].plot(history['val_acc'], label='Validation', linewidth=2)
axes[1].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
_add_phase(axes[1])
axes[1].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# F1 Score
axes[2].plot(history['train_f1'], label='Train', linewidth=2)
axes[2].plot(history['val_f1'], label='Validation', linewidth=2)
axes[2].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
_add_phase(axes[2])
axes[2].set_title('Weighted F1 Score', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

# Learning Rate
axes[3].plot(history['lr'], linewidth=2, color='purple')
_add_phase(axes[3])
axes[3].set_title('Learning Rate', fontsize=14, fontweight='bold')
axes[3].set_xlabel('Epoch')
axes[3].set_yscale('log')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'motor_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()


## 7️⃣ Test Set Evaluation

Load the best checkpoint and evaluate on the held-out test set.
Since this is a binary task, we get:
- **Accuracy**, **F1**, **Sensitivity** (true positive rate for PD), **Specificity** (true negative rate)
- **ROC-AUC** — area under the receiver operating characteristic curve
- **Confusion matrix** — shows false positives vs false negatives

In [ ]:
# ============================================================
# CELL 7: Test Set Evaluation
# ============================================================

# Load best model
checkpoint = torch.load(os.path.join(MODEL_DIR, 'motor_best.pt'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']} (val F1={checkpoint['val_f1']:.4f})")

# Evaluate on test set
test_loss, test_acc, test_f1, test_preds, test_labels, test_probs = validate(
    model, test_loader, criterion, device
)

print(f"\n{'='*60}")
print(f"📊 TEST SET RESULTS ({len(df_test_orig)} images)")
print(f"{'='*60}")
print(f"   Accuracy:      {test_acc:.4f} ({test_acc*100:.1f}%)")
print(f"   F1 (weighted): {test_f1:.4f}")
print(f"   Loss:          {test_loss:.4f}")

# Per-class report
class_names_list = ['Healthy', "Parkinson's"]
print(f"\n📋 Classification Report:")
print(classification_report(test_labels, test_preds, target_names=class_names_list, digits=3))

# ROC-AUC
try:
    auc = roc_auc_score(test_labels, test_probs[:, 1])
    print(f"   ROC-AUC: {auc:.4f}")
except Exception as e:
    print(f"   ROC-AUC: Could not compute ({e})")

# Sensitivity & Specificity
cm = confusion_matrix(test_labels, test_preds)
if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    print(f"\n   Clinical Metrics:")
    print(f"   Sensitivity (PD detection): {sensitivity:.3f} ({sensitivity*100:.1f}%)")
    print(f"   Specificity (Healthy correct): {specificity:.3f} ({specificity*100:.1f}%)")
    print(f"   PPV (Positive Predictive Value): {ppv:.3f}")
    print(f"   NPV (Negative Predictive Value): {npv:.3f}")

# ═══════════════════════════════════════════════════════════
# Confusion Matrix
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_list, yticklabels=class_names_list, ax=axes[0],
            annot_kws={"size": 18})
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=class_names_list, yticklabels=class_names_list, ax=axes[1],
            annot_kws={"size": 18})
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'motor_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════
# Separate evaluation by drawing type
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"📊 PER-DRAWING-TYPE EVALUATION")
print(f"{'='*60}")

for drawing_type in df_test_orig['drawing_type'].unique():
    df_type_test = df_test_orig[df_test_orig['drawing_type'] == drawing_type]
    if len(df_type_test) < 5:
        continue
    
    type_dataset = ParkinsonsDrawingDataset(df_type_test, transform=val_transforms)
    type_loader = DataLoader(type_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    _, type_acc, type_f1, type_preds, type_labels, type_probs = validate(
        model, type_loader, criterion, device
    )
    
    type_auc = roc_auc_score(type_labels, type_probs[:, 1]) if len(np.unique(type_labels)) > 1 else 0
    
    print(f"\n   {drawing_type.upper()} ({len(df_type_test)} images):")
    print(f"     Accuracy: {type_acc:.3f} | F1: {type_f1:.3f} | AUC: {type_auc:.3f}")

# ═══════════════════════════════════════════════════════════
# Separate evaluation by dataset source
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"📊 PER-DATASET EVALUATION")
print(f"{'='*60}")

for dataset_name in df_test_orig['dataset'].unique():
    df_ds_test = df_test_orig[df_test_orig['dataset'] == dataset_name]
    if len(df_ds_test) < 5:
        continue
    
    ds_dataset = ParkinsonsDrawingDataset(df_ds_test, transform=val_transforms)
    ds_loader = DataLoader(ds_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    _, ds_acc, ds_f1, ds_preds, ds_labels, ds_probs = validate(
        model, ds_loader, criterion, device
    )
    
    ds_auc = roc_auc_score(ds_labels, ds_probs[:, 1]) if len(np.unique(ds_labels)) > 1 else 0
    
    print(f"\n   {dataset_name} ({len(df_ds_test)} images):")
    print(f"     Accuracy: {ds_acc:.3f} | F1: {ds_f1:.3f} | AUC: {ds_auc:.3f}")

## 8️⃣ Explainable AI (XAI) — Multi-Method Analysis

We apply **5 complementary XAI techniques** to explain model decisions for clinical interpretability:

| Method | Library | What it reveals |
|--------|---------|----------------|
| **GradCAM** | pytorch-grad-cam | Class-discriminative heatmap from final conv layer |
| **GradCAM++** | pytorch-grad-cam | Better localization for multiple tremor regions |
| **LIME** | lime | Superpixel-based local explanations |
| **Integrated Gradients** | captum | Pixel attribution via gradient path integral |
| **Occlusion Sensitivity** | captum | Systematic occlusion to find important regions |

### Why multi-XAI matters for clinical applications:
- Different methods highlight **different aspects** of tremor/micrographia
- Consensus across methods → **higher confidence** in explanation
- Doctors can compare views to validate diagnosis rationale

In [ ]:
# ============================================================
# CELL 8a: GradCAM & GradCAM++ Implementation
# ============================================================

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

class MotorModelWrapper(nn.Module):
    """Wrapper that returns only logits (GradCAM needs single output)."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, x):
        logits, _ = self.model(x)
        return logits

wrapped_model = MotorModelWrapper(model)

# Target layer — last convolutional layer of EfficientNet-B0
target_layer = model.backbone.conv_head

# Initialize both CAM methods
cam_gradcam = GradCAM(model=wrapped_model, target_layers=[target_layer])
cam_gradcampp = GradCAMPlusPlus(model=wrapped_model, target_layers=[target_layer])

# ═══════════════════════════════════════════════════════════
# Unified function for both GradCAM variants
# ═══════════════════════════════════════════════════════════
def generate_cam(image_path, true_label=None, method='gradcam'):
    """Generate GradCAM or GradCAM++ for a drawing image."""
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        logits, risk = model(img_tensor)
        pred_class = logits.argmax(dim=1).item()
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        risk_val = risk.item()
    
    targets = [ClassifierOutputTarget(pred_class)]
    cam_obj = cam_gradcam if method == 'gradcam' else cam_gradcampp
    grayscale_cam = cam_obj(input_tensor=img_tensor, targets=targets)[0]
    
    img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_resized).astype(np.float32) / 255.0
    cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    
    return {
        'original': img_np,
        'cam_overlay': cam_image,
        'heatmap': grayscale_cam,
        'pred_class': pred_class,
        'true_label': true_label,
        'probs': probs,
        'risk': risk_val,
        'method': method,
    }

# ═══════════════════════════════════════════════════════════
# GradCAM vs GradCAM++ Comparison Grid
# ═══════════════════════════════════════════════════════════
print("🔍 Generating GradCAM & GradCAM++ visualizations...")

columns = []
for dt in df_test_orig['drawing_type'].unique():
    for cls in ['healthy', 'parkinson']:
        subset = df_test_orig[(df_test_orig['drawing_type'] == dt) & (df_test_orig['class_name'] == cls)]
        if len(subset) > 0:
            columns.append((dt, cls, f'{dt.title()} — {cls.title()}'))

n_cols = min(len(columns), 6)

if n_cols > 0:
    fig, axes = plt.subplots(4, n_cols, figsize=(4 * n_cols, 16))
    fig.suptitle('GradCAM vs GradCAM++ Comparison\n'
                 '(Red = High Attention for PD detection | Blue = Low Attention)',
                 fontsize=16, fontweight='bold')
    
    if n_cols == 1:
        axes = axes.reshape(4, 1)
    
    for col_idx in range(n_cols):
        drawing_type, class_name, title = columns[col_idx]
        subset = df_test_orig[(df_test_orig['drawing_type'] == drawing_type) & (df_test_orig['class_name'] == class_name)]
        
        if len(subset) == 0:
            for row in range(4):
                axes[row, col_idx].axis('off')
            continue
        
        sample = subset.sample(1, random_state=SEED + col_idx).iloc[0]
        r_gc = generate_cam(sample['path'], true_label=sample['label'], method='gradcam')
        r_gcpp = generate_cam(sample['path'], true_label=sample['label'], method='gradcam++')
        
        # Row 0: Original
        axes[0, col_idx].imshow(r_gc['original'])
        axes[0, col_idx].set_title(title, fontsize=10, fontweight='bold',
                                    color='green' if class_name == 'healthy' else 'red')
        axes[0, col_idx].axis('off')
        
        # Row 1: GradCAM
        axes[1, col_idx].imshow(r_gc['cam_overlay'])
        pred_name = CLASS_NAMES[r_gc['pred_class']]
        correct = r_gc['pred_class'] == sample['label']
        axes[1, col_idx].set_title(f"GradCAM — {pred_name} ({'✓' if correct else '✗'})", fontsize=9,
                                    color='green' if correct else 'red')
        axes[1, col_idx].axis('off')
        
        # Row 2: GradCAM++
        axes[2, col_idx].imshow(r_gcpp['cam_overlay'])
        axes[2, col_idx].set_title(f"GradCAM++ — Risk: {r_gcpp['risk']:.2f}", fontsize=9)
        axes[2, col_idx].axis('off')
        
        # Row 3: Heatmap difference
        diff = np.abs(r_gc['heatmap'] - r_gcpp['heatmap'])
        axes[3, col_idx].imshow(diff, cmap='hot')
        axes[3, col_idx].set_title(f"Difference (Δ={diff.mean():.3f})", fontsize=9)
        axes[3, col_idx].axis('off')
    
    for i, label in enumerate(['Original', 'GradCAM', 'GradCAM++', 'Difference']):
        axes[i, 0].set_ylabel(label, fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'gradcam_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ GradCAM & GradCAM++ visualizations saved!")
else:
    print("⚠️  No test images found for visualization")

In [ ]:
# ============================================================
# CELL 11: GradCAM — All Test Samples + Misclassified Analysis
# ============================================================

# Find correct and misclassified examples
correct_mask = test_preds == test_labels
correct_indices = np.where(correct_mask)[0]
wrong_indices = np.where(~correct_mask)[0]

print(f"   Correct: {len(correct_indices)} | Misclassified: {len(wrong_indices)}")
print(f"   Test Accuracy: {len(correct_indices)/(len(correct_indices)+len(wrong_indices)):.1%}")

if len(wrong_indices) > 0:
    n_show = min(6, len(wrong_indices))
    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
    fig.suptitle('GradCAM on MISCLASSIFIED Samples\n(What confused the model?)',
                 fontsize=14, fontweight='bold')
    
    if n_show == 1:
        axes = axes.reshape(2, 1)
    
    for i, idx in enumerate(wrong_indices[:n_show]):
        row = df_test_orig.iloc[idx]
        result = generate_gradcam(row['path'], true_label=int(test_labels[idx]))
        
        # Original
        axes[0, i].imshow(result['original'])
        axes[0, i].set_title(
            f"True: {CLASS_NAMES[int(test_labels[idx])]}\n"
            f"Pred: {CLASS_NAMES[int(test_preds[idx])]}",
            fontsize=9, color='red', fontweight='bold'
        )
        axes[0, i].axis('off')
        
        # GradCAM
        axes[1, i].imshow(result['gradcam'])
        axes[1, i].set_title(
            f"{row['drawing_type'].upper()}\n"
            f"Risk: {result['risk']:.2f} | Conf: {result['probs'][result['pred_class']]:.1%}",
            fontsize=8
        )
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'motor_misclassified_gradcam.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("🎉 No misclassifications on test set!")

### 8b. LIME — Local Interpretable Model-agnostic Explanations

LIME (Ribeiro et al., 2016) explains individual predictions by:
1. Segmenting the image into **superpixels**
2. Randomly perturbing (turning off) superpixels
3. Training a local linear model on perturbations → predictions
4. Highlighting superpixels that **most influence** the prediction

**For spiral drawings**: LIME shows which drawing segments (curved lines, intersections, tremor zones) drive the PD prediction.

In [ ]:
# ============================================================
# CELL 8b: LIME — Superpixel Explanations
# ============================================================

from lime import lime_image
from skimage.segmentation import mark_boundaries

# LIME prediction function — must return probabilities for all classes
def lime_predict_fn(images_np):
    """Batch prediction for LIME. Input: (N, H, W, 3) numpy uint8/float."""
    model.eval()
    batch = []
    for img in images_np:
        if img.max() <= 1.0:
            img = (img * 255).astype(np.uint8)
        pil_img = Image.fromarray(img)
        tensor = val_transforms(pil_img)
        batch.append(tensor)
    
    batch_tensor = torch.stack(batch).to(device)
    with torch.no_grad():
        logits, _ = model(batch_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    return probs

# Initialize LIME explainer
lime_explainer = lime_image.LimeImageExplainer(random_state=SEED)

# ═══════════════════════════════════════════════════════════
# Generate LIME explanations for test samples
# ═══════════════════════════════════════════════════════════
print("🔍 Generating LIME explanations (this may take 1-2 minutes)...")

n_lime = min(4, len(df_test_orig))
lime_samples = df_test_orig.groupby('class_name').apply(
    lambda x: x.sample(min(2, len(x)), random_state=SEED)
).reset_index(drop=True)

fig, axes = plt.subplots(3, n_lime, figsize=(5 * n_lime, 15))
fig.suptitle('LIME Explanations — Which drawing regions drive the prediction?',
             fontsize=16, fontweight='bold')

if n_lime == 1:
    axes = axes.reshape(3, 1)

for i, (_, sample) in enumerate(lime_samples.head(n_lime).iterrows()):
    img_pil = Image.open(sample['path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_pil)
    
    # LIME explanation
    explanation = lime_explainer.explain_instance(
        img_np, lime_predict_fn,
        top_labels=2,
        hide_color=0,
        num_samples=500,          # Balance between quality and speed
        batch_size=32,
        segmentation_fn=None      # Uses quickshift by default
    )
    
    # Get mask for predicted class
    model.eval()
    with torch.no_grad():
        tensor = val_transforms(img_pil).unsqueeze(0).to(device)
        logits, risk = model(tensor)
        pred_class = logits.argmax(dim=1).item()
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    
    # Row 0: Original
    axes[0, i].imshow(img_np)
    true_cls = 'Healthy' if sample['label'] == 0 else "Parkinson's"
    axes[0, i].set_title(f"True: {true_cls} ({sample['drawing_type']})",
                          fontsize=10, fontweight='bold',
                          color='green' if sample['label'] == 0 else 'red')
    axes[0, i].axis('off')
    
    # Row 1: LIME — positive regions (supporting prediction)
    temp, mask = explanation.get_image_and_mask(
        pred_class, positive_only=True, num_features=8, hide_rest=False, min_weight=0.01
    )
    axes[1, i].imshow(mark_boundaries(temp / 255.0, mask))
    pred_name = CLASS_NAMES[pred_class]
    axes[1, i].set_title(f"LIME (+): {pred_name} ({probs[pred_class]:.1%})",
                          fontsize=9, color='green' if pred_class == sample['label'] else 'red')
    axes[1, i].axis('off')
    
    # Row 2: LIME — positive + negative regions
    temp2, mask2 = explanation.get_image_and_mask(
        pred_class, positive_only=False, num_features=10, hide_rest=False, min_weight=0.01
    )
    axes[2, i].imshow(mark_boundaries(temp2 / 255.0, mask2))
    axes[2, i].set_title(f"LIME (±): Green=FOR, Red=AGAINST", fontsize=9)
    axes[2, i].axis('off')

for i, label in enumerate(['Original', 'LIME (positive)', 'LIME (positive+negative)']):
    axes[i, 0].set_ylabel(label, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'lime_explanations.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ LIME explanations saved!")

### 8c. Integrated Gradients — Pixel-Level Attribution (Captum)

Integrated Gradients (Sundararajan et al., 2017) computes **per-pixel importance** by:
1. Interpolating between a **baseline** (black image) and the input
2. Computing gradients at each interpolation step
3. Accumulating gradients along the path → pixel attribution map

**For spiral drawings**: Shows exactly which pixels (pen strokes, tremor marks) contributed to the prediction.

In [ ]:
# ============================================================
# CELL 8c: Integrated Gradients (Captum)
# ============================================================

from captum.attr import IntegratedGradients, visualization as captum_viz

# Wrap model for Captum — needs single tensor output per class
class CaptumWrapper(nn.Module):
    """Wrapper for Captum: returns logits only."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, x):
        logits, _ = self.model(x)
        return logits

captum_model = CaptumWrapper(model)
captum_model.eval()

ig = IntegratedGradients(captum_model)

# ═══════════════════════════════════════════════════════════
# Generate IG attributions
# ═══════════════════════════════════════════════════════════
print("🔍 Computing Integrated Gradients (50 steps per image)...")

n_ig = min(4, len(df_test_orig))
ig_samples = df_test_orig.groupby('class_name').apply(
    lambda x: x.sample(min(2, len(x)), random_state=SEED)
).reset_index(drop=True).head(n_ig)

fig, axes = plt.subplots(3, n_ig, figsize=(5 * n_ig, 15))
fig.suptitle('Integrated Gradients — Pixel-Level Attributions\n'
             '(Green = supports prediction, Red = opposes)',
             fontsize=16, fontweight='bold')

if n_ig == 1:
    axes = axes.reshape(3, 1)

for i, (_, sample) in enumerate(ig_samples.iterrows()):
    img_pil = Image.open(sample['path']).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    img_tensor.requires_grad = True
    
    # Get prediction
    with torch.no_grad():
        logits, _ = model(img_tensor)
        pred_class = logits.argmax(dim=1).item()
    
    # Integrated Gradients
    baseline = torch.zeros_like(img_tensor).to(device)  # black baseline
    attributions = ig.attribute(
        img_tensor,
        baselines=baseline,
        target=pred_class,
        n_steps=50,
        return_convergence_delta=False
    )
    
    # Convert to numpy for visualization
    attr_np = attributions.squeeze().cpu().detach().numpy()
    attr_np = np.transpose(attr_np, (1, 2, 0))  # CHW → HWC
    
    # Original image (denormalized)
    img_show = img_tensor.squeeze().cpu().detach().numpy()
    img_show = np.transpose(img_show, (1, 2, 0))
    img_show = img_show * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_show = np.clip(img_show, 0, 1)
    
    # Row 0: Original
    axes[0, i].imshow(img_show)
    true_cls = 'Healthy' if sample['label'] == 0 else "Parkinson's"
    axes[0, i].set_title(f"True: {true_cls} ({sample['drawing_type']})",
                          fontsize=10, fontweight='bold',
                          color='green' if sample['label'] == 0 else 'red')
    axes[0, i].axis('off')
    
    # Row 1: IG attribution magnitude (absolute)
    attr_magnitude = np.abs(attr_np).sum(axis=2)
    attr_magnitude = (attr_magnitude - attr_magnitude.min()) / (attr_magnitude.max() - attr_magnitude.min() + 1e-8)
    axes[1, i].imshow(img_show)
    axes[1, i].imshow(attr_magnitude, cmap='hot', alpha=0.6)
    pred_name = CLASS_NAMES[pred_class]
    correct = pred_class == sample['label']
    axes[1, i].set_title(f"IG (|attr|): {pred_name} ({'✓' if correct else '✗'})",
                          fontsize=9, color='green' if correct else 'red')
    axes[1, i].axis('off')
    
    # Row 2: IG attribution sign (positive=green, negative=red)
    attr_signed = attr_np.sum(axis=2)
    pos = np.clip(attr_signed, 0, None)
    neg = np.clip(-attr_signed, 0, None)
    pos_norm = pos / (pos.max() + 1e-8)
    neg_norm = neg / (neg.max() + 1e-8)
    
    rgb_overlay = np.zeros((*attr_signed.shape, 3))
    rgb_overlay[:, :, 1] = pos_norm  # Green = positive
    rgb_overlay[:, :, 0] = neg_norm  # Red = negative
    
    axes[2, i].imshow(img_show)
    axes[2, i].imshow(rgb_overlay, alpha=0.5)
    axes[2, i].set_title("IG (signed): Green=FOR, Red=AGAINST", fontsize=9)
    axes[2, i].axis('off')

for i, label in enumerate(['Original', 'IG Magnitude', 'IG Signed']):
    axes[i, 0].set_ylabel(label, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'integrated_gradients.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Integrated Gradients visualizations saved!")

### 8d. Occlusion Sensitivity — Systematic Region Importance (Captum)

Occlusion Sensitivity (Zeiler & Fergus, 2014) measures importance by:
1. Sliding a **gray patch** across the image
2. Recording how the prediction **changes** when each region is hidden
3. Regions where occlusion drops confidence → **important for classification**

**For spiral drawings**: Reveals which drawing segments are essential — if hiding a tremor region doesn't change the prediction, that region wasn't used.

In [ ]:
# ============================================================
# CELL 8d: Occlusion Sensitivity (Captum)
# ============================================================

from captum.attr import Occlusion

occlusion = Occlusion(captum_model)

print("🔍 Computing Occlusion Sensitivity (sliding window 15×15, stride 8)...")

n_occ = min(4, len(df_test_orig))
occ_samples = df_test_orig.groupby('class_name').apply(
    lambda x: x.sample(min(2, len(x)), random_state=SEED)
).reset_index(drop=True).head(n_occ)

fig, axes = plt.subplots(2, n_occ, figsize=(5 * n_occ, 10))
fig.suptitle('Occlusion Sensitivity — Which regions are essential?\n'
             '(Bright = prediction drops when occluded → important region)',
             fontsize=16, fontweight='bold')

if n_occ == 1:
    axes = axes.reshape(2, 1)

for i, (_, sample) in enumerate(occ_samples.iterrows()):
    img_pil = Image.open(sample['path']).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits, _ = model(img_tensor)
        pred_class = logits.argmax(dim=1).item()
    
    # Occlusion attribution
    attr = occlusion.attribute(
        img_tensor,
        target=pred_class,
        strides=(3, 8, 8),          # (C, H, W) stride
        sliding_window_shapes=(3, 15, 15),  # (C, H, W) patch size
        baselines=0                  # Gray occlusion patch
    )
    
    attr_np = attr.squeeze().cpu().detach().numpy()
    attr_np = np.transpose(attr_np, (1, 2, 0))  # CHW → HWC
    
    # Denormalized original
    img_show = img_tensor.squeeze().cpu().detach().numpy()
    img_show = np.transpose(img_show, (1, 2, 0))
    img_show = img_show * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_show = np.clip(img_show, 0, 1)
    
    # Row 0: Original
    axes[0, i].imshow(img_show)
    true_cls = 'Healthy' if sample['label'] == 0 else "Parkinson's"
    pred_name = CLASS_NAMES[pred_class]
    correct = pred_class == sample['label']
    axes[0, i].set_title(f"True: {true_cls} | Pred: {pred_name} ({'✓' if correct else '✗'})",
                          fontsize=10, fontweight='bold',
                          color='green' if correct else 'red')
    axes[0, i].axis('off')
    
    # Row 1: Occlusion heatmap
    occ_magnitude = np.abs(attr_np).sum(axis=2)
    occ_magnitude = (occ_magnitude - occ_magnitude.min()) / (occ_magnitude.max() - occ_magnitude.min() + 1e-8)
    axes[1, i].imshow(img_show)
    axes[1, i].imshow(occ_magnitude, cmap='jet', alpha=0.6)
    axes[1, i].set_title(f"Occlusion Map ({sample['drawing_type']})", fontsize=9)
    axes[1, i].axis('off')

for i, label in enumerate(['Original', 'Occlusion Sensitivity']):
    axes[i, 0].set_ylabel(label, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'occlusion_sensitivity.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Occlusion Sensitivity maps saved!")

### 8e. XAI Comparison Grid — All 5 Methods Side by Side

Final visualization comparing **all 5 XAI methods** on the same images.
This is the key figure for the NeuroVerse XAI module — shows doctors that **multiple independent methods agree** on which drawing regions indicate PD.

In [ ]:
# ============================================================
# CELL 8e: XAI Comparison Grid — All 5 Methods on Same Images
# ============================================================

print("🔍 Generating XAI comparison grid (all 5 methods)...")

# Pick 1 healthy + 1 PD sample
compare_samples = []
for cls in [0, 1]:
    sub = df_test_orig[df_test_orig['label'] == cls]
    if len(sub) > 0:
        compare_samples.append(sub.sample(1, random_state=SEED + cls).iloc[0])

n_samples = len(compare_samples)
methods = ['Original', 'GradCAM', 'GradCAM++', 'LIME', 'Integrated\nGradients', 'Occlusion\nSensitivity']

fig, axes = plt.subplots(n_samples, 6, figsize=(30, 5 * n_samples))
fig.suptitle('XAI Comparison: 5 Methods Applied to the Same Spiral/Wave Drawings\n'
             '(Consistent highlighting across methods = reliable explanation)',
             fontsize=18, fontweight='bold', y=1.02)

if n_samples == 1:
    axes = axes.reshape(1, 6)

for row_idx, sample in enumerate(compare_samples):
    img_pil = Image.open(sample['path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(img_pil).astype(np.float32) / 255.0
    img_tensor = val_transforms(Image.open(sample['path']).convert('RGB')).unsqueeze(0).to(device)
    
    # Get prediction
    model.eval()
    with torch.no_grad():
        logits, risk = model(img_tensor)
        pred_class = logits.argmax(dim=1).item()
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    
    true_cls = 'Healthy' if sample['label'] == 0 else "Parkinson's"
    pred_cls = CLASS_NAMES[pred_class]
    correct = pred_class == sample['label']
    
    # Col 0: Original
    axes[row_idx, 0].imshow(img_np)
    axes[row_idx, 0].set_title(f"True: {true_cls}\nPred: {pred_cls} ({'✓' if correct else '✗'})",
                                fontsize=11, fontweight='bold',
                                color='green' if correct else 'red')
    axes[row_idx, 0].axis('off')
    
    # Col 1: GradCAM
    targets = [ClassifierOutputTarget(pred_class)]
    gc_map = cam_gradcam(input_tensor=img_tensor, targets=targets)[0]
    gc_overlay = show_cam_on_image(img_np, gc_map, use_rgb=True)
    axes[row_idx, 1].imshow(gc_overlay)
    axes[row_idx, 1].set_title('GradCAM', fontsize=11, fontweight='bold')
    axes[row_idx, 1].axis('off')
    
    # Col 2: GradCAM++
    gcpp_map = cam_gradcampp(input_tensor=img_tensor, targets=targets)[0]
    gcpp_overlay = show_cam_on_image(img_np, gcpp_map, use_rgb=True)
    axes[row_idx, 2].imshow(gcpp_overlay)
    axes[row_idx, 2].set_title('GradCAM++', fontsize=11, fontweight='bold')
    axes[row_idx, 2].axis('off')
    
    # Col 3: LIME
    img_uint8 = (img_np * 255).astype(np.uint8)
    lime_exp = lime_explainer.explain_instance(
        img_uint8, lime_predict_fn, top_labels=2,
        hide_color=0, num_samples=300, batch_size=32
    )
    temp, mask = lime_exp.get_image_and_mask(
        pred_class, positive_only=True, num_features=8, hide_rest=False, min_weight=0.01
    )
    axes[row_idx, 3].imshow(mark_boundaries(temp / 255.0, mask))
    axes[row_idx, 3].set_title('LIME', fontsize=11, fontweight='bold')
    axes[row_idx, 3].axis('off')
    
    # Col 4: Integrated Gradients
    img_tensor_ig = img_tensor.clone().requires_grad_(True)
    baseline = torch.zeros_like(img_tensor_ig).to(device)
    ig_attr = ig.attribute(img_tensor_ig, baselines=baseline, target=pred_class, n_steps=50)
    ig_np = ig_attr.squeeze().cpu().detach().numpy()
    ig_np = np.transpose(ig_np, (1, 2, 0))
    ig_mag = np.abs(ig_np).sum(axis=2)
    ig_mag = (ig_mag - ig_mag.min()) / (ig_mag.max() - ig_mag.min() + 1e-8)
    axes[row_idx, 4].imshow(img_np)
    axes[row_idx, 4].imshow(ig_mag, cmap='hot', alpha=0.6)
    axes[row_idx, 4].set_title('Integrated\nGradients', fontsize=11, fontweight='bold')
    axes[row_idx, 4].axis('off')
    
    # Col 5: Occlusion
    occ_attr = occlusion.attribute(
        img_tensor, target=pred_class,
        strides=(3, 8, 8), sliding_window_shapes=(3, 15, 15), baselines=0
    )
    occ_np = occ_attr.squeeze().cpu().detach().numpy()
    occ_np = np.transpose(occ_np, (1, 2, 0))
    occ_mag = np.abs(occ_np).sum(axis=2)
    occ_mag = (occ_mag - occ_mag.min()) / (occ_mag.max() - occ_mag.min() + 1e-8)
    axes[row_idx, 5].imshow(img_np)
    axes[row_idx, 5].imshow(occ_mag, cmap='jet', alpha=0.6)
    axes[row_idx, 5].set_title('Occlusion\nSensitivity', fontsize=11, fontweight='bold')
    axes[row_idx, 5].axis('off')

# Add column headers
for j, method in enumerate(methods):
    axes[0, j].annotate(method, xy=(0.5, 1.15), xycoords='axes fraction',
                         ha='center', fontsize=12, fontweight='bold', color='navy')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'xai_comparison_grid.png'), dpi=200, bbox_inches='tight')
plt.show()

print("✅ XAI Comparison Grid saved!")
print(f"\n📊 XAI Summary:")
print(f"   Methods used: GradCAM, GradCAM++, LIME, Integrated Gradients, Occlusion Sensitivity")
print(f"   All visualizations saved to: {MODEL_DIR}")
print(f"\n   For doctor dashboard: use GradCAM (fastest) + LIME (most interpretable)")
print(f"   For research paper: use all 5 methods to demonstrate robustness")

## 9️⃣ Full Fine-tuning (Unfreeze All Layers)

After the head is well-trained, unfreeze ALL backbone layers for a few more epochs at very low LR.
This typically adds **2-5% accuracy** on small datasets — the backbone layers adapt to the specific drawing domain.

In [ ]:
# ============================================================
# CELL 12: Full Fine-tuning (Optional)
# ============================================================

FULL_FINETUNE = True  # Set False to skip

if FULL_FINETUNE:
    print("🔓 Unfreezing ALL backbone layers for full fine-tuning...")
    
    for param in model.parameters():
        param.requires_grad = True
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Trainable parameters: {total_params:,}")
    
    FT_LR = 5e-6       # Very conservative for small dataset
    FT_EPOCHS = 20
    FT_PATIENCE = 10
    
    ft_optimizer = optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=1e-4)
    ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=FT_EPOCHS, eta_min=1e-8)
    
    print(f"   LR: {FT_LR} | Epochs: {FT_EPOCHS} | Patience: {FT_PATIENCE}")
    print()
    
    ft_patience_counter = 0
    
    for epoch in range(FT_EPOCHS):
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, criterion, ft_optimizer, scaler, device
        )
        val_loss, val_acc, val_f1, _, _, _ = validate(model, val_loader, criterion, device)
        ft_scheduler.step()
        
        print(f"  FT Epoch {epoch+1:2d}/{FT_EPOCHS} │ "
              f"Train: acc={train_acc:.3f} f1={train_f1:.3f} │ "
              f"Val: acc={val_acc:.3f} f1={val_f1:.3f}", end="")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            ft_patience_counter = 0
            torch.save({
                'epoch': NUM_EPOCHS + epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': ft_optimizer.state_dict(),
                'val_f1': val_f1,
                'val_acc': val_acc,
                'num_classes': NUM_CLASSES,
                'img_size': IMG_SIZE,
                'class_names': CLASS_NAMES,
                'full_finetuned': True,
            }, os.path.join(MODEL_DIR, 'motor_best.pt'))
            print(f" ★ NEW BEST (f1={val_f1:.4f})")
        else:
            ft_patience_counter += 1
            print()
        
        if ft_patience_counter >= FT_PATIENCE:
            print(f"\n⏹️  Fine-tuning stopped (no improvement for {FT_PATIENCE} epochs)")
            break
    
    print(f"\n✅ Fine-tuning complete! Best F1: {best_val_f1:.4f}")
    
    # Reload best
    checkpoint = torch.load(os.path.join(MODEL_DIR, 'motor_best.pt'), map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    print("⏭️  Skipping full fine-tuning")

## 🔟 Model Export for NeuroVerse Backend

Export the trained model in formats for deployment:
1. **`motor_spiral_model.pt`** — Full PyTorch model for backend (`app/ml/models/`)
2. **`motor_spiral_model_mobile.pt`** — TorchScript for mobile
3. **`motor_config.json`** — Class mapping + UPDRS thresholds for `fusion_service.py`

### How it maps to UPDRS (MDS-UPDRS 3.15-3.18):
| Prediction | UPDRS Tremor | Clinical Meaning |
|-----------|-------------|-----------------|
| Healthy (conf > 0.90) | 0 | Normal |
| Healthy (conf 0.70-0.90) | 1 | Slight tremor possible |
| PD (conf 0.50-0.70) | 2 | Mild tremor |
| PD (conf 0.70-0.85) | 3 | Moderate tremor |
| PD (conf > 0.85) | 4 | Severe tremor |

In [ ]:
# ============================================================
# CELL 11: Model Export
# ============================================================

EXPORT_DIR = os.path.join(OUTPUT_DIR, "export")
os.makedirs(EXPORT_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════
# 1. Full PyTorch model (for backend)
# ═══════════════════════════════════════════════════════════
export_path = os.path.join(EXPORT_DIR, "motor_spiral_model.pt")

torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'architecture': 'efficientnet_b0',
        'num_classes': NUM_CLASSES,
        'img_size': IMG_SIZE,
        'dropout': 0.5,
    },
    'class_mapping': {
        'class_names': CLASS_NAMES,
        'label_to_class': {0: 'healthy', 1: 'parkinson'},
    },
    'normalization': {
        'mean': IMAGENET_MEAN,
        'std': IMAGENET_STD,
    },
    'metrics': {
        'test_accuracy': float(test_acc),
        'test_f1': float(test_f1),
        'best_val_f1': float(best_val_f1),
    },
    'clinical_thresholds': {
        # Maps model confidence → UPDRS tremor score (3.15-3.18)
        'updrs_mapping': {
            'healthy_high_conf':   {'min_conf': 0.90, 'updrs_tremor': 0, 'label': 'Normal'},
            'healthy_moderate':    {'min_conf': 0.70, 'updrs_tremor': 1, 'label': 'Slight'},
            'pd_low_conf':         {'min_conf': 0.50, 'updrs_tremor': 2, 'label': 'Mild tremor'},
            'pd_moderate_conf':    {'min_conf': 0.70, 'updrs_tremor': 3, 'label': 'Moderate tremor'},
            'pd_high_conf':        {'min_conf': 0.85, 'updrs_tremor': 4, 'label': 'Severe tremor'},
        },
        # Risk mapping for fusion_service.py
        'pd_risk_by_prediction': {
            'healthy': {'pd_risk': 0.05, 'ad_risk': 0.02},
            'parkinson': {'pd_risk': 0.65, 'ad_risk': 0.05},
        },
        # Tremor frequency range (PD-characteristic)
        'pd_tremor_frequency_hz': [4.0, 6.0],
        # Tapping threshold from fusion_service.py
        'tapping_pd_threshold': 3.0,
    },
    'dataset_info': {
        'name': 'YOLODatasetFull + HandPD_Augmented_Data (multi-source)',
        'sources': [
            'YOLODatasetFull (spiral + wave drawings, 3,732 images)',
            'HandPD_Augmented_Data — Geometric (Pereira et al., 2016)',
            'HandPD_Augmented_Data — Mixed (Pereira et al., 2016)',
            'HandPD_Augmented_Data — Photometric (Pereira et al., 2016)',
        ],
        'total_images': len(df_all),
        'drawing_types': sorted(df_all['drawing_type'].unique().tolist()),
        'papers': [
            'Pereira et al. (2016) — ESWA',
            'Zham et al. (2017) — Frontiers in Neurology',
        ],
    }
}, export_path)

size_mb = os.path.getsize(export_path) / (1024 * 1024)
print(f"✅ Full model exported: {export_path} ({size_mb:.1f} MB)")

# ═══════════════════════════════════════════════════════════
# 2. TorchScript model (mobile / ONNX)
# ═══════════════════════════════════════════════════════════
try:
    model.eval()
    
    class MotorInference(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.backbone = model.backbone
            self.classifier = model.classifier
        
        def forward(self, x):
            features = self.backbone(x)
            logits = self.classifier(features)
            return logits
    
    inference_model = MotorInference(model)
    scripted = torch.jit.trace(inference_model, torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device))
    script_path = os.path.join(EXPORT_DIR, "motor_spiral_model_mobile.pt")
    scripted.save(script_path)
    size_mb_s = os.path.getsize(script_path) / (1024 * 1024)
    print(f"✅ TorchScript model: {script_path} ({size_mb_s:.1f} MB)")
except Exception as e:
    print(f"⚠️  TorchScript export failed (non-critical): {e}")

# ═══════════════════════════════════════════════════════════
# 3. Config JSON
# ═══════════════════════════════════════════════════════════
config = {
    "model_name": "MotorNet",
    "version": "2.0",
    "architecture": "efficientnet_b0",
    "num_classes": NUM_CLASSES,
    "img_size": IMG_SIZE,
    "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
    "class_names": CLASS_NAMES,
    "drawing_types": sorted(df_all['drawing_type'].unique().tolist()),
    "updrs_mapping": {
        "0": {"pd_risk": 0.05, "updrs_tremor": 0, "label": "Normal — No tremor"},
        "1": {"pd_risk": 0.65, "updrs_tremor": 2, "label": "PD detected — Tremor present"},
    },
    "confidence_to_updrs": [
        {"pred": "healthy", "conf_min": 0.90, "updrs": 0, "severity": "Normal"},
        {"pred": "healthy", "conf_min": 0.70, "updrs": 1, "severity": "Slight"},
        {"pred": "parkinson", "conf_min": 0.50, "updrs": 2, "severity": "Mild"},
        {"pred": "parkinson", "conf_min": 0.70, "updrs": 3, "severity": "Moderate"},
        {"pred": "parkinson", "conf_min": 0.85, "updrs": 4, "severity": "Severe"},
    ],
    "metrics": {
        "test_accuracy": round(float(test_acc), 4),
        "test_f1": round(float(test_f1), 4),
        "best_val_f1": round(float(best_val_f1), 4),
    },
    "dataset": {
        "total_images": len(df_all),
        "sources": ["YOLODatasetFull", "HandPD_Augmented_Geometric",
                     "HandPD_Augmented_Mixed", "HandPD_Augmented_Photometric"],
    }
}

config_path = os.path.join(EXPORT_DIR, "motor_config.json")
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✅ Config JSON: {config_path}")

# ═══════════════════════════════════════════════════════════
# Summary
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"📦 EXPORTED FILES:")
print(f"{'='*60}")
for f in os.listdir(EXPORT_DIR):
    fpath = os.path.join(EXPORT_DIR, f)
    fsize = os.path.getsize(fpath) / (1024 * 1024)
    print(f"   {f:45s} {fsize:8.2f} MB")
print(f"\n   Copy these to your backend:")
print(f"   motor_spiral_model.pt  → neuroverse-backend/app/ml/models/")
print(f"   motor_config.json      → neuroverse-backend/app/ml/config/")

## 1️⃣1️⃣ Backend Integration Code

Ready-to-copy Python code for `neuroverse-backend/app/ml/predictors/motor_predictor.py`.
This handles:
1. Loading the trained model
2. Predicting PD risk from spiral/wave drawings
3. Mapping to UPDRS tremor scores
4. Integrating with `fusion_service.py` motor assessment

In [ ]:
# ============================================================
# CELL 14: Backend Integration Code (copy to your backend)
# ============================================================

backend_code = '''
# ═══════════════════════════════════════════════════════════
# FILE: neuroverse-backend/app/ml/predictors/motor_predictor.py
# ═══════════════════════════════════════════════════════════
"""
Motor Drawing Predictor for NeuroVerse Backend.
Predicts PD risk from spiral/wave drawings using trained MotorNet.
Maps to UPDRS tremor scores for fusion_service.py integration.
"""
import os
import json
import torch
import torch.nn as nn
import timm
import numpy as np
from PIL import Image
from torchvision import transforms


class MotorNet(nn.Module):
    """Motor drawing classifier — must match training architecture."""
    
    def __init__(self, num_classes=2, dropout=0.5):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b0', pretrained=False, num_classes=0, global_pool='avg'
        )
        feature_dim = self.backbone.num_features  # 1280
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_dim, 256), nn.ReLU(inplace=True), nn.BatchNorm1d(256),
            nn.Dropout(dropout * 0.6),
            nn.Linear(256, 64), nn.ReLU(inplace=True), nn.BatchNorm1d(64),
            nn.Linear(64, num_classes),
        )
        self.risk_head = nn.Sequential(
            nn.Linear(num_classes, 16), nn.ReLU(inplace=True),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    
    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        with torch.no_grad():
            probs = torch.softmax(logits, dim=-1)
        risk = self.risk_head(probs)
        return logits, risk.squeeze(-1)


class MotorPredictor:
    """Motor drawing predictor for NeuroVerse backend."""
    
    # Confidence → UPDRS tremor score mapping
    UPDRS_MAP = [
        # (predicted_class, min_confidence, updrs_score, label)
        ('healthy', 0.90, 0, 'Normal'),
        ('healthy', 0.70, 1, 'Slight tremor possible'),
        ('parkinson', 0.50, 2, 'Mild tremor'),
        ('parkinson', 0.70, 3, 'Moderate tremor'),
        ('parkinson', 0.85, 4, 'Severe tremor'),
    ]
    
    def __init__(self, model_path: str = None):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        if model_path is None:
            model_path = os.path.join(
                os.path.dirname(__file__), '..', 'models', 'motor_spiral_model.pt'
            )
        
        checkpoint = torch.load(model_path, map_location=self.device)
        config = checkpoint['model_config']
        
        self.model = MotorNet(
            num_classes=config['num_classes'],
            dropout=config['dropout']
        ).to(self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        
        self.img_size = config['img_size']
        norm = checkpoint['normalization']
        self.transform = transforms.Compose([
            transforms.Resize((self.img_size, self.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=norm['mean'], std=norm['std']),
        ])
        
        self.class_names = checkpoint['class_mapping']['class_names']
        self.clinical = checkpoint['clinical_thresholds']
        print(f"✅ Motor model loaded ({self.device})")
    
    def predict(self, image_path: str, drawing_type: str = 'spiral') -> dict:
        """
        Predict PD risk from a spiral/wave drawing.
        
        Returns dict with:
            - predicted_class: 'healthy' or 'parkinson'
            - confidence: float [0, 1]
            - pd_risk: float [0, 1]
            - updrs_tremor: int 0-4 (UPDRS 3.15-3.18)
            - severity_label: str
            - spiral_tremor: float [0, 1] (for fusion_service.py)
        """
        img = Image.open(image_path).convert('RGB')
        tensor = self.transform(img).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            logits, risk = self.model(tensor)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            pred_idx = int(logits.argmax(dim=1).item())
            confidence = float(probs[pred_idx])
        
        pred_class = 'healthy' if pred_idx == 0 else 'parkinson'
        
        # Map to UPDRS tremor score
        updrs_tremor = 0
        severity_label = 'Normal'
        
        if pred_class == 'parkinson':
            if confidence >= 0.85:
                updrs_tremor, severity_label = 4, 'Severe tremor'
            elif confidence >= 0.70:
                updrs_tremor, severity_label = 3, 'Moderate tremor'
            else:
                updrs_tremor, severity_label = 2, 'Mild tremor'
        else:
            if confidence >= 0.90:
                updrs_tremor, severity_label = 0, 'Normal'
            else:
                updrs_tremor, severity_label = 1, 'Slight tremor possible'
        
        # Compute spiral_tremor (0-1 scale) for fusion_service.py
        # Maps to existing: spiral_tremor feature used in _assess_motor()
        spiral_tremor = probs[1]  # PD probability = tremor severity proxy
        
        pd_risk_info = self.clinical['pd_risk_by_prediction'].get(pred_class, {})
        
        return {
            'predicted_class': pred_class,
            'confidence': confidence,
            'class_probabilities': {str(i): float(p) for i, p in enumerate(probs)},
            'pd_risk': pd_risk_info.get('pd_risk', 0.0) * (confidence if pred_class == 'parkinson' else 1.0),
            'ad_risk': pd_risk_info.get('ad_risk', 0.0),
            'updrs_tremor': updrs_tremor,
            'severity_label': severity_label,
            'drawing_type': drawing_type,
            # Features for fusion_service.py _assess_motor():
            'spiral_tremor': float(spiral_tremor),
            'tremor_frequency': 5.0 if pred_class == 'parkinson' and confidence > 0.70 else 0.0,
            'is_pd': pred_class == 'parkinson',
        }
    
    def predict_from_bytes(self, image_bytes: bytes, drawing_type: str = 'spiral') -> dict:
        """Predict from raw image bytes (for API endpoint)."""
        import io, tempfile
        img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
            img.save(f.name)
            result = self.predict(f.name, drawing_type)
            os.unlink(f.name)
        return result


# ═══════════════════════════════════════════════════════════
# Usage in FastAPI:
# ═══════════════════════════════════════════════════════════
#
# predictor = MotorPredictor()
#
# @router.post("/predict/spiral-drawing")
# async def predict_spiral(file: UploadFile, drawing_type: str = "spiral"):
#     image_bytes = await file.read()
#     result = predictor.predict_from_bytes(image_bytes, drawing_type)
#     return result
#
# # The result feeds directly into fusion_service.py:
# # features = {
# #     "spiral_tremor": result["spiral_tremor"],      # → UPDRS 3.15-3.18
# #     "tremor_frequency": result["tremor_frequency"], # → PD 4-6 Hz check
# #     "spiral_duration": 10.0,                        # from app timer
# # }
'''

print(backend_code)
print("\n" + "=" * 60)
print("📋 Copy the code above to:")
print("   neuroverse-backend/app/ml/predictors/motor_predictor.py")
print("=" * 60)

## 1️⃣2️⃣ Inference Demo — Predict & Visualize

In [ ]:
# ============================================================
# CELL 13: Inference Demo — Predict & Visualize
# ============================================================

def predict_and_visualize(image_path, drawing_type='spiral', show_gradcam=True):
    """Full inference pipeline with GradCAM."""
    
    img_pil = Image.open(image_path).convert('RGB')
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        logits, risk = model(img_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = logits.argmax(dim=1).item()
        risk_val = risk.item()
    
    pred_class = CLASS_NAMES[pred_idx]
    confidence = probs[pred_idx]
    
    # UPDRS mapping
    if pred_idx == 1:  # PD
        if confidence >= 0.85:
            updrs, severity = 4, 'Severe'
        elif confidence >= 0.70:
            updrs, severity = 3, 'Moderate'
        else:
            updrs, severity = 2, 'Mild'
    else:  # Healthy
        if confidence >= 0.90:
            updrs, severity = 0, 'Normal'
        else:
            updrs, severity = 1, 'Slight'
    
    if show_gradcam:
        targets = [ClassifierOutputTarget(pred_idx)]
        grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0]
        img_resized = img_pil.resize((IMG_SIZE, IMG_SIZE))
        img_np = np.array(img_resized).astype(np.float32) / 255.0
        cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
        
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        
        # Original
        axes[0].imshow(img_pil)
        axes[0].set_title(f'Original ({drawing_type})', fontsize=12, fontweight='bold')
        axes[0].axis('off')
        
        # GradCAM
        axes[1].imshow(cam_image)
        axes[1].set_title('GradCAM — Tremor Regions', fontsize=12, fontweight='bold')
        axes[1].axis('off')
        
        # Probability bars
        colors = ['#4ECDC4', '#FF6B6B']
        bars = axes[2].barh(
            list(CLASS_NAMES.values()),
            [probs[i] for i in range(NUM_CLASSES)],
            color=colors
        )
        axes[2].set_xlim(0, 1)
        axes[2].set_title('Class Probabilities', fontsize=12, fontweight='bold')
        for bar, p in zip(bars, probs):
            if p > 0.05:
                axes[2].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                           f'{p:.1%}', va='center', fontsize=11)
        
        title_color = 'green' if pred_idx == 0 else 'red'
        plt.suptitle(
            f"Prediction: {pred_class} (UPDRS {updrs} — {severity}) | PD Risk: {risk_val:.2f}",
            fontsize=14, fontweight='bold', color=title_color
        )
        plt.tight_layout()
        plt.show()
    
    return {
        'predicted_class': pred_class,
        'confidence': float(confidence),
        'pd_risk': float(risk_val),
        'updrs_tremor': updrs,
        'severity': severity,
        'drawing_type': drawing_type,
        'probabilities': {CLASS_NAMES[i]: float(probs[i]) for i in range(NUM_CLASSES)}
    }


# ═══════════════════════════════════════════════════════════
# Test on samples from each drawing type + class
# ═══════════════════════════════════════════════════════════
print("🔍 Running inference tests...\n")

for drawing_type in df_test_orig['drawing_type'].unique():
    for class_name in ['healthy', 'parkinson']:
        subset = df_test_orig[
            (df_test_orig['drawing_type'] == drawing_type) &
            (df_test_orig['class_name'] == class_name)
        ]
        if len(subset) == 0:
            continue
        
        sample = subset.sample(1, random_state=SEED).iloc[0]
        print(f"{'='*60}")
        print(f"🔍 {drawing_type.upper()} — True: {class_name.upper()}")
        print(f"{'='*60}")
        result = predict_and_visualize(sample['path'], drawing_type)
        print(f"   Result: {result['predicted_class']} | UPDRS: {result['updrs_tremor']} | "
              f"Conf: {result['confidence']:.1%} | Risk: {result['pd_risk']:.2f}")
        print()

## 1️⃣3️⃣ K-Fold Cross-Validation (Robustness Check)

Since the dataset is small (~102 images), single train/test split results can be noisy.
5-fold cross-validation gives a more reliable estimate of true model performance.

In [ ]:
# ============================================================
# CELL 16: 5-Fold Cross-Validation
# ============================================================

from sklearn.model_selection import StratifiedKFold

RUN_KFOLD = True  # Set False to skip (takes ~5x training time)
K_FOLDS = 5
KFOLD_EPOCHS = 30  # Fewer epochs per fold

if RUN_KFOLD:
    print(f"{'='*60}")
    print(f"🔄 {K_FOLDS}-Fold Cross-Validation")
    print(f"{'='*60}")
    
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
    fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(df_all, df_all['label'])):
        print(f"\n── Fold {fold+1}/{K_FOLDS} ──")
        
        df_fold_train = df_all.iloc[train_idx]
        df_fold_val = df_all.iloc[val_idx]
        
        # Create datasets with oversampling
        fold_train_ds = ParkinsonsDrawingDataset(df_fold_train, transform=train_transforms, oversample_factor=OVERSAMPLE)
        fold_val_ds = ParkinsonsDrawingDataset(df_fold_val, transform=val_transforms)
        
        # Sampler
        fold_counts = df_fold_train['label'].value_counts().sort_index()
        fold_weights = 1.0 / fold_counts.values
        fold_sample_weights = np.tile(
            df_fold_train['label'].map(dict(zip(fold_counts.index, fold_weights))).values,
            OVERSAMPLE
        )
        fold_sampler = WeightedRandomSampler(fold_sample_weights, len(fold_sample_weights), replacement=True)
        
        fold_train_loader = DataLoader(fold_train_ds, batch_size=BATCH_SIZE, sampler=fold_sampler, num_workers=2, pin_memory=True, drop_last=True)
        fold_val_loader = DataLoader(fold_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
        
        # Fresh model
        fold_model = MotorNet(num_classes=2, pretrained=True, dropout=0.5).to(device)
        
        fold_bb_params = [p for n, p in fold_model.backbone.named_parameters() if p.requires_grad]
        fold_head_params = list(fold_model.classifier.parameters()) + list(fold_model.risk_head.parameters())
        fold_optimizer = optim.AdamW([
            {'params': fold_bb_params, 'lr': LEARNING_RATE / 10},
            {'params': fold_head_params, 'lr': LEARNING_RATE},
        ], weight_decay=WEIGHT_DECAY)
        fold_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(fold_optimizer, T_0=10, T_mult=2, eta_min=1e-7)
        fold_scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
        
        fold_best_f1 = 0
        fold_patience = 0
        
        for ep in range(KFOLD_EPOCHS):
            train_one_epoch(fold_model, fold_train_loader, criterion, fold_optimizer, fold_scaler, device)
            _, val_acc, val_f1, _, _, _ = validate(fold_model, fold_val_loader, criterion, device)
            fold_scheduler.step()
            
            if val_f1 > fold_best_f1:
                fold_best_f1 = val_f1
                fold_best_acc = val_acc
                fold_patience = 0
            else:
                fold_patience += 1
            
            if fold_patience >= 10:
                break
        
        # Final evaluation
        _, fold_acc, fold_f1, fold_preds, fold_labels, fold_probs = validate(
            fold_model, fold_val_loader, criterion, device
        )
        
        try:
            fold_auc = roc_auc_score(fold_labels, fold_probs[:, 1])
        except:
            fold_auc = 0.0
        
        fold_results.append({
            'fold': fold + 1,
            'accuracy': fold_best_acc,
            'f1': fold_best_f1,
            'auc': fold_auc,
        })
        
        print(f"   Fold {fold+1}: Acc={fold_best_acc:.3f} | F1={fold_best_f1:.3f} | AUC={fold_auc:.3f}")
        
        del fold_model, fold_optimizer
        torch.cuda.empty_cache() if device.type == 'cuda' else None
    
    # Summary
    df_folds = pd.DataFrame(fold_results)
    print(f"\n{'='*60}")
    print(f"📊 {K_FOLDS}-FOLD CROSS-VALIDATION RESULTS")
    print(f"{'='*60}")
    print(f"   Accuracy: {df_folds['accuracy'].mean():.3f} ± {df_folds['accuracy'].std():.3f}")
    print(f"   F1 Score: {df_folds['f1'].mean():.3f} ± {df_folds['f1'].std():.3f}")
    print(f"   ROC-AUC:  {df_folds['auc'].mean():.3f} ± {df_folds['auc'].std():.3f}")
    print(f"\n   Per-fold:")
    for _, r in df_folds.iterrows():
        print(f"     Fold {int(r['fold'])}: Acc={r['accuracy']:.3f} F1={r['f1']:.3f} AUC={r['auc']:.3f}")
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(K_FOLDS)
    width = 0.25
    ax.bar(x - width, df_folds['accuracy'], width, label='Accuracy', color='#4ECDC4')
    ax.bar(x, df_folds['f1'], width, label='F1', color='#FF6B6B')
    ax.bar(x + width, df_folds['auc'], width, label='AUC', color='#45B7D1')
    ax.set_xlabel('Fold')
    ax.set_ylabel('Score')
    ax.set_title(f'{K_FOLDS}-Fold Cross-Validation Results', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'Fold {i+1}' for i in range(K_FOLDS)])
    ax.legend()
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, 'motor_kfold_results.png'), dpi=150, bbox_inches='tight')
    plt.show()

else:
    print("⏭️  Skipping K-Fold cross-validation")

---

## ✅ Summary — Motor Spiral & Wave Drawing Model Complete!

### What was trained:
| Component | Details |
|-----------|---------|
| **Model** | MotorNet (EfficientNet-B0 + custom PD head) |
| **Dataset** | 12,300+ real images + DDPM synthetic data from 2 sources |
| **Task** | Binary: Healthy vs Parkinson's Disease |
| **Drawing Types** | Spiral + Wave only (meander → separate notebook) |
| **Augmentation** | 12 traditional transforms + **DDPM diffusion generation** + weighted sampling |
| **Synthetic Data** | Conditional DDPM (cosine schedule, 500 steps) — balances minority class |
| **Training** | AdamW + Cosine Annealing Warm Restarts + Label Smoothing |
| **XAI** | 5 methods: GradCAM, GradCAM++, LIME, IG, Occlusion |
| **Validation** | 5-Fold cross-validation for robustness |

### Data Augmentation Pipeline:
| Stage | Method | Purpose |
|-------|--------|---------|
| **1. DDPM Synthesis** | Conditional Denoising Diffusion (64×64 → 224×224) | Generate novel minority-class images |
| **2. Traditional Augmentation** | 12 transforms (rotation, flip, perspective, color jitter, etc.) | Online per-batch diversity |
| **3. Weighted Sampling** | Inverse class-frequency weights | Balance mini-batches |

### XAI Methods Used:
| Method | Library | Purpose |
|--------|---------|---------|
| GradCAM | pytorch-grad-cam | Fast class-discriminative heatmap |
| GradCAM++ | pytorch-grad-cam | Better multi-region localization |
| LIME | lime | Superpixel-based local explanations |
| Integrated Gradients | captum | Pixel-level attribution via path integral |
| Occlusion Sensitivity | captum | Systematic region importance mapping |

### Dataset sources:
| Dataset | Images | Classes | Source |
|---------|--------|---------|--------|
| YOLODatasetFull | ~3,732 | Spiral + Wave (H/PD) | YOLO format |
| HandPD Geometric | ~2,170 | Spiral (H/PD) | Pereira et al. 2016 |
| HandPD Mixed | ~4,260 | Spiral (H/PD) | Pereira et al. 2016 |
| HandPD Photometric | ~2,090 | Spiral (H/PD) | Pereira et al. 2016 |
| **DDPM Synthetic** | **Auto-balanced** | Minority class | Generated in-notebook |
| **TOTAL** | **~12,300+ real + synthetic** | Binary (Healthy=0, PD=1) | Combined |

### Exported files:
| File | Destination | Purpose |
|------|------------|---------|
| `motor_spiral_model.pt` | `app/ml/models/` | Backend inference |
| `motor_spiral_model_mobile.pt` | — | Mobile/edge deployment |
| `motor_config.json` | `app/ml/config/` | UPDRS mapping + thresholds |
| `ddpm_spiral_best.pt` | — | Trained diffusion model (for future synthesis) |

### How it fits in NeuroVerse:
```
Flutter App: User draws spiral/wave on Canvas
    ↓ PNG image + stroke data (speed, pressure, tremor)
FastAPI Backend: MotorPredictor.predict(image)
    ↓ {predicted_class: "parkinson", pd_risk: 0.72, updrs_tremor: 3}
fusion_service.py _assess_motor():
    ↓ Combines spiral_tremor + tapping_rate + tremor_frequency
    ↓ Calculates Hoehn & Yahr stage
    ↓ UPDRS motor total score
xai_service.py: Multi-XAI explanations
    ↓ GradCAM + LIME for doctor dashboard
    ↓ All 5 methods for detailed analysis
```

### References:
1. Pereira et al. (2016) — HandPD spiral dataset (*ESWA*)
2. Ho et al. (2020) — *"Denoising Diffusion Probabilistic Models"* (NeurIPS)
3. Nichol & Dhariwal (2021) — *"Improved DDPM"* (ICML)
4. Selvaraju et al. (2017) — GradCAM
5. Ribeiro et al. (2016) — LIME
6. Sundararajan et al. (2017) — Integrated Gradients

### NeuroVerse Detection Modules:
- [x] **Speech** — DementiaBank + EWA-DB ✅
- [x] **Clock Drawing (CDT)** — Roboflow CDT ✅ (87.2% accuracy)
- [x] **Motor/Spiral** — YOLODatasetFull + HandPD Spiral + DDPM Synthetic ✅ (THIS NOTEBOOK)
- [ ] **Motor/Meander** — HandPD Meander + DDPM Synthetic → `motor_meander_training.ipynb`
- [ ] **Gait** — Walking + Balance + Turn-in-Place
- [ ] **Fusion** — Combine all module scores

In [ ]:

# ╔══════════════════════════════════════════════════════════════════╗
# ║  Save All Spiral Artifacts → Google Drive  Neuro_Models/spiral/ ║
# ╚══════════════════════════════════════════════════════════════════╝
#
#   Neuro_Models/
#   └── spiral/
#       ├── models/    ← .pt (full + mobile), motor_config.json
#       ├── plots/     ← all training/eval/XAI PNGs
#       └── configs/   ← training results JSON

import shutil

SPIRAL_DRIVE_ROOT = "/content/drive/MyDrive/Neuro_Models/spiral"
SUB_MODELS  = os.path.join(SPIRAL_DRIVE_ROOT, "models")
SUB_PLOTS   = os.path.join(SPIRAL_DRIVE_ROOT, "plots")
SUB_CONFIGS = os.path.join(SPIRAL_DRIVE_ROOT, "configs")

for folder in [SUB_MODELS, SUB_PLOTS, SUB_CONFIGS]:
    os.makedirs(folder, exist_ok=True)

print("📁 Drive folder structure created:")
print(f"   Neuro_Models/spiral/")
print(f"   ├── models/")
print(f"   ├── plots/")
print(f"   └── configs/")

uploaded = []
missing = []

# ══════════════════════════════════════════════════════════════
# 1. Models — from EXPORT_DIR
# ══════════════════════════════════════════════════════════════
MODEL_FILES = [
    "motor_spiral_model.pt",
    "motor_spiral_model_mobile.pt",
    "motor_config.json",
]

for fname in MODEL_FILES:
    src = os.path.join(EXPORT_DIR, fname)
    if os.path.exists(src):
        dst = os.path.join(SUB_MODELS, fname)
        shutil.copy2(src, dst)
        size_kb = os.path.getsize(dst) / 1024
        print(f"   ✅ [models]  {fname:45s} {size_kb:8.1f} KB")
        uploaded.append(fname)
    else:
        missing.append(fname)

# ══════════════════════════════════════════════════════════════
# 2. DDPM model (if trained)
# ══════════════════════════════════════════════════════════════
ddpm_src = "ddpm_spiral_best.pt"
if os.path.exists(ddpm_src):
    dst = os.path.join(SUB_MODELS, ddpm_src)
    shutil.copy2(ddpm_src, dst)
    size_kb = os.path.getsize(dst) / 1024
    print(f"   ✅ [models]  {ddpm_src:45s} {size_kb:8.1f} KB")
    uploaded.append(ddpm_src)
elif os.path.exists(os.path.join(MODEL_DIR, ddpm_src)):
    dst = os.path.join(SUB_MODELS, ddpm_src)
    shutil.copy2(os.path.join(MODEL_DIR, ddpm_src), dst)
    size_kb = os.path.getsize(dst) / 1024
    print(f"   ✅ [models]  {ddpm_src:45s} {size_kb:8.1f} KB")
    uploaded.append(ddpm_src)

# ══════════════════════════════════════════════════════════════
# 3. Plots — from MODEL_DIR (checkpoints folder)
# ══════════════════════════════════════════════════════════════
PLOT_FILES = [
    "motor_training_curves.png",
    "motor_confusion_matrix.png",
    "gradcam_comparison.png",
    "motor_misclassified_gradcam.png",
    "lime_explanations.png",
    "integrated_gradients.png",
    "occlusion_sensitivity.png",
    "xai_comparison_grid.png",
    "motor_kfold_results.png",
]

copied_plots = 0
for pname in PLOT_FILES:
    src = os.path.join(MODEL_DIR, pname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(SUB_PLOTS, pname))
        copied_plots += 1
        uploaded.append(pname)
    else:
        # Also check OUTPUT_DIR root
        src2 = os.path.join(OUTPUT_DIR, pname)
        if os.path.exists(src2):
            shutil.copy2(src2, os.path.join(SUB_PLOTS, pname))
            copied_plots += 1
            uploaded.append(pname)
        else:
            missing.append(pname)

print(f"   ✅ [plots]   {copied_plots}/{len(PLOT_FILES)} plot PNGs copied")

# ══════════════════════════════════════════════════════════════
# 4. Training results JSON
# ══════════════════════════════════════════════════════════════
results = {
    "model": "MotorNet (EfficientNet-B0)",
    "task": "Binary — Healthy vs Parkinson's",
    "test_accuracy": round(float(test_acc), 4) if 'test_acc' in dir() else None,
    "test_f1": round(float(test_f1), 4) if 'test_f1' in dir() else None,
    "best_val_f1": round(float(best_val_f1), 4) if 'best_val_f1' in dir() else None,
    "dataset_size": len(df_all) if 'df_all' in dir() else None,
    "drawing_types": sorted(df_all['drawing_type'].unique().tolist()) if 'df_all' in dir() else None,
    "class_names": CLASS_NAMES if 'CLASS_NAMES' in dir() else None,
    "img_size": IMG_SIZE if 'IMG_SIZE' in dir() else 224,
    "sources": [
        "YOLODatasetFull (spiral + wave)",
        "HandPD Geometric (Pereira et al. 2016)",
        "HandPD Mixed (Pereira et al. 2016)",
        "HandPD Photometric (Pereira et al. 2016)",
        "DDPM Synthetic (generated in-notebook)",
    ],
}

# Add K-fold results if available
if 'df_folds' in dir():
    results["cv_accuracy_mean"] = round(float(df_folds['accuracy'].mean()), 4)
    results["cv_accuracy_std"]  = round(float(df_folds['accuracy'].std()), 4)
    results["cv_f1_mean"]       = round(float(df_folds['f1'].mean()), 4)
    results["cv_auc_mean"]      = round(float(df_folds['auc'].mean()), 4)

results_path = os.path.join(SUB_CONFIGS, "spiral_training_results.json")
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"   ✅ [configs] spiral_training_results.json saved")
uploaded.append("spiral_training_results.json")

# ══════════════════════════════════════════════════════════════
# Summary
# ══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"📦 All Spiral artifacts saved to Google Drive!")
print(f"   📂 {SPIRAL_DRIVE_ROOT}")
print(f"   ├── models/   → {len([f for f in uploaded if f in MODEL_FILES + [ddpm_src]])} files")
print(f"   ├── plots/    → {copied_plots} visualisation PNGs")
print(f"   └── configs/  → spiral_training_results.json")
print(f"\n   ✅ Uploaded: {len(uploaded)} files")
if missing:
    print(f"   ⚠️  Missing:  {len(missing)} files (run those cells first):")
    for m in missing:
        print(f"      • {m}")
print(f"{'='*60}")
